In [ ]:
import numpy as np
from matplotlib import pyplot as plt
from sympy import *
from scipy.fft import fft, ifft,fftfreq, fftshift, fft2,  ifftshift
from scipy import special
import math
from scipy.interpolate import griddata
from scipy.interpolate import interp1d
from imageio import imwrite
from imageio import imread
from matplotlib.gridspec import GridSpec
import gc
from scipy.fft import fft, ifft,fftfreq, fftshift, fft2,  ifftshift
from scipy import special
from scipy.optimize import curve_fit
from scipy import ndimage
from scipy.signal import find_peaks
from scipy.ndimage import map_coordinates
from scipy.signal import find_peaks
from scipy.ndimage import gaussian_filter
from imageio import imread
from scipy.optimize import curve_fit

from scipy.stats import skewnorm
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap, BoundaryNorm
from scipy.optimize import least_squares
from numba import njit, prange
import math
import numpy as np
# import sys
# sys.path.append('/Users/elenaros/Documents/WARG/Code/Optics/EOS/EOS_cleaned_up/EOS_funcs_2.py')
# import EOS_funcs as eos

from imageio import imwrite
from matplotlib import rc
import matplotlib.patches as patches

rc('font',**{'family':'serif','serif':['Times']})
rc('text', usetex=False)

In [ ]:
25-12


In [ ]:
6.60+19.50+5

In [ ]:
115e-12* 3e8 *1e3 /2

In [ ]:
""" 

Parameters and Constants

"""

######################################################################################################
#Constants
######################################################################################################
c = 2.99792458e8 #m/s
hbar = 1.055e-34 #J s
e0 = 8.85e-12 #F/m


######################################################################################################
#Laser Parameters
######################################################################################################
l_laser = 800e-9 #m
alpha_ld = 5 # Laser angle of incidence (deg)/ Cannot be zero!
alpha_l = alpha_ld * np.pi/ 180 #Laser angle of incidence (rad)
Epulse = 100e-6 # Laser pulse energy (J)
dlaser = 10e-3 # Laser diameter (transverse FWHM) (m)
sig_tr_laser = dlaser/2 # Transverse width assuiming flat-top (m)
FWHMlaser = 100e-15 #Laser pulse length (s)
Ilaser_inc = Epulse/(FWHMlaser * np.pi * (dlaser/2)**2) # Laser Incident Intensity per Pulse (W/m^2)
sig_0_laser = FWHMlaser/(2* np.sqrt(2 * np.log(2))) # Assumes gaussian pulse (s)
sig_eff_laser = sig_0_laser/np.cos(alpha_l) # accounts for angle of incidence (s)

######################################################################################################
#Parameters for Crystal
######################################################################################################
# crystal thickness (m):
d25 = 25e-6 
d50  = 50e-6
d75 = 75e-6
d100 = 100e-6
d150 = 150e-6
d200 = 200e-6
dcry = d100 #choosing total crystal thickness

######################################################################################################
#Parameters for Ebeam for 2 Bunch and Single
######################################################################################################
#Drive Bunch
Q1 = 1.6e-9 # Charge (C)
sig_z1_thz = 40e-6 # length (m)
sig_t1_thz = sig_z1_thz/c #temporal length (s)

#Witness Bunch
Q2 = 0e-12 # Charge (C)
sig_z2_thz = 35e-6 # length (m)
sig_t2_thz = sig_z2_thz/c #temporal length (s)

#Two-Bunch Seperation
delz12_thz = 135e-6 # length (m)
delt12_thz = delz12_thz/c # temporal (s)

#Single bunch
Q0 = 3e-9 # charge (C)
sig_z_thz = 35e-6 # length (m)
sig_t_thz = sig_z_thz/c #temporal (s)



#### Defining Functions

In [ ]:
def build_4crystal_geometry(X, Y, Ri, crystal_size=10e-3,
                            delay_topbottom=True,
                            delay_amount=200e-15):
    """
    Builds 4 overlapping 10 mm crystal regions.

    Top/Bottom → rotated 90°
    Left/Right → not rotated

    Top/Bottom optionally delayed.

    Returns:
        geometry dict containing masks + rotation + delay
    """

    Ri_m = Ri * 1e-3
    half = crystal_size / 2

    geometry = {}

    # -------------------------
    # Horizontal crystals
    # -------------------------
    geometry["top"] = {}
    geometry["top"]["mask"] = (Y >= Ri_m - half) & (Y <= Ri_m + half)
    geometry["top"]["rotated"] = True
    geometry["top"]["delay"] = delay_amount if delay_topbottom else 0

    geometry["bottom"] = {}
    geometry["bottom"]["mask"] = (Y >= -Ri_m - half) & (Y <= -Ri_m + half)
    geometry["bottom"]["rotated"] = True
    geometry["bottom"]["delay"] = delay_amount if delay_topbottom else 0

    # -------------------------
    # Vertical crystals
    # -------------------------
    geometry["left"] = {}
    geometry["left"]["mask"] = (X >= -Ri_m - half) & (X <= -Ri_m + half)
    geometry["left"]["rotated"] = False
    geometry["left"]["delay"] = 0

    geometry["right"] = {}
    geometry["right"]["mask"] = (X >= Ri_m - half) & (X <= Ri_m + half)
    geometry["right"]["rotated"] = False
    geometry["right"]["delay"] = 0

    return geometry

In [ ]:
def plot_crystal_geometry(X, Y, geometry):
    crystal_map = np.zeros_like(X)

    crystal_map[geometry["top"]["mask"]] = 1
    crystal_map[geometry["bottom"]["mask"]] = 2
    crystal_map[geometry["left"]["mask"]] = 3
    crystal_map[geometry["right"]["mask"]] = 4

    plt.figure(figsize=(6,6))
    plt.imshow(crystal_map,
               origin='lower',
               extent=[X.min()*1e3, X.max()*1e3,
                       Y.min()*1e3, Y.max()*1e3],
               aspect='equal')
    plt.colorbar(label="1=Top, 2=Bottom, 3=Left, 4=Right")
    plt.xlabel("x (mm)")
    plt.ylabel("y (mm)")
    plt.title("4-Crystal Geometry")
    plt.show()

def plot_overlap_geometry(x, y, m_top, m_bot, m_left, m_right, title="Crystal Geometry"):
    """
    Visualize 4 rectangle masks.
    0=none, 1=top, 2=bottom, 3=left, 4=right.
    If overlaps exist, later assignments overwrite earlier ones (just for display).
    """
    geo = np.zeros(m_top.shape, dtype=np.int8)

    geo[m_top] = 1
    geo[m_bot] = 2
    geo[m_left] = 3
    geo[m_right] = 4

    cmap = ListedColormap(["black", "tab:blue", "tab:green", "tab:orange", "tab:red"])

    plt.figure(figsize=(6, 5), constrained_layout=True)
    plt.imshow(
        geo,
        origin="lower",
        extent=[x.min()*1e3, x.max()*1e3, y.min()*1e3, y.max()*1e3],
        aspect="equal",
        cmap=cmap,
        vmin=0, vmax=4
    )

In [ ]:
######################################################################################################
#Constants
######################################################################################################
c = 2.99792458e8 #m/s
hbar = 1.055e-34 #J s
e0 = 8.85e-12 #F/m


def FThz1d(t2d, E2d):
    """
    Take the center-row (uniform t) and return (f, E_f).
    Assumes t2d, E2d have shape (ny, nx) with t2d increasing along x.
    """
    ny, nx = t2d.shape
    cy = ny // 2
    t = t2d[cy, :]                # shape (nx,)
    E = E2d[cy, :]                # shape (nx,)
    dt = t[1] - t[0]

    # FFT (no extra factors; use numpy convention)
    Ef = fftshift(fft(ifftshift(E))) *dt #* np.sqrt(2*np.pi)
    f  = fftshift(fftfreq(nx, d=dt))   # Hz

    return f, Ef, dt

def ifft_response(f, Ef, dt):
    """
    Performs inverse FFT of Ef (frequency-domain field) back to time domain.
    Assumes Ef was created with fft(ifftshift(E_t)) * dt * sqrt(2π)
    """
    Ef_resp = Ef  # (you can multiply by a response function here if needed)
    
    # Do not shadow numpy.fft.ifft
    e_t = fftshift(ifft(ifftshift(Ef_resp))) / dt #(np.sqrt(2 * np.pi) * dt)
    
    return e_t.real                  

def recast_1d_to_2d(Rl, E_1d):
    """
    Map a 1D radial profile E_1d (defined along the center row, from the center to +x)
    back onto the 2D radius map Rl.

    Assumes:
      - E_1d has length len(x) (full row).
      - We will use only the center -> +x half where the radius is monotonic.
    """
    ny, nx = Rl.shape
    cy = ny // 2
    cx = nx // 2

    # monotonic radii along center row (from center to right edge)
    r_uniform = Rl[cy, cx:]          # shape (nx - cx,)
    e_uniform = E_1d[cx:]            # match length

    # ensure strictly increasing x for interp1d (deduplicate if needed)
    r_unique, keep = np.unique(r_uniform, return_index=True)
    e_unique = e_uniform[keep]

    interp_E = interp1d(r_unique, e_unique, bounds_error=False, fill_value=0.0)
    return interp_E(Rl)

######################################################################################################
# THz Field Approximation for Single Bunch
######################################################################################################

def E0_approx(r, Q0, sig_t_thz):
   return (Q0 / (2 * np.pi * e0 * r *c)) * (1/(np.sqrt(2 * np.pi) * sig_t_thz)) 

def Er_s(r,t,sig_t_thz, time_delay):
   return E0_approx(r) * np.exp(- (t-time_delay)**2 / (2 * sig_t_thz**2)) 


######################################################################################################
# THz Field Approximation for Two Bunch
######################################################################################################

def E01_approx(r,Q1, sig_t1_thz ): 
   return (Q1 / (2 * np.pi * e0 * r*c)) * (1/(np.sqrt(2 * np.pi) * sig_t1_thz)) #fix this

def E02_approx(r,Q2, sig_t2_thz ):
   return (Q2 / (2 * np.pi * e0 * r*c)) * (1/(np.sqrt(2 * np.pi) * sig_t2_thz))

def Er_2(r,t, time_delay, delt12_thz,sig_t2_thz , sig_t1_thz, Q1, Q2):
   return  E02_approx(r,Q2, sig_t2_thz ) * np.exp(- (t-time_delay-delt12_thz)**2 / (2 * sig_t2_thz**2)) + E01_approx(r,Q1, sig_t1_thz) * np.exp(- (t-time_delay)**2 / (2 * sig_t1_thz**2))

def Er2t(r2,t,time_delay,delt12_thz,sig_t2_thz,Q2):
   return  E02_approx(r2,Q2, sig_t2_thz) * np.exp(- (t-time_delay-delt12_thz)**2 / (2 * sig_t2_thz**2)) 

def Er1t(r1,t,time_delay,sig_t1_thz,Q1):
   return  E01_approx(r1,Q1, sig_t1_thz) * np.exp(- (t-time_delay)**2 / (2 * sig_t1_thz**2))


######################################################################################################
#Index of Refraction (Casalbouoni PRSTAB 2008)
######################################################################################################
eps_el = 8.7
S0 = 1.8
fr0 = 10.98e12 #(Hz)
Del0 = 0.02e12 #(Hz)

def eps(f):
    return eps_el + (S0 * fr0**2)/(fr0**2 -f**2-complex(0,Del0)*f)
def n(f):
    return np.real(np.sqrt(eps(f)))
def k(f):
    return np.imag(np.sqrt(eps(f)))


######################################################################################################
#Group Velocity of GaP for 800 nm Laser (Casalbouoni PRSTAB 2008)
######################################################################################################
# here x is the wavelength of the laser
def nopt(x):
    return np.sqrt(2.680 + (6.40 *(x *1e6)**2)/((x *1e6)**2 - 0.0903279))

def dnopt(x):
    return 2529822.1281347*(-1.0e-12*x**3/(x**2 - 9.03279e-14)**2 + 1.0*x/(1000000000000.0*x**2 - 0.0903279))/(x**2/(1000000000000.0*x**2 - 0.0903279) + 4.1875e-13)**0.5

def vg_opt(x):
    return (c/nopt(x)) *(1 + (x/nopt(x))*dnopt(x))

def vg_opt_eff(x, alpha_l):
    return vg_opt(x) * np.cos(alpha_l)

######################################################################################################
#Group Velocity and Phase Velocity of GaP for THz (Casalbouoni PRSTAB 2008)
######################################################################################################
#f is the frequency of the Thz pulse
def vg(f):
    droote = 5.06756393339777e-40*(2*f + complex(0,20000000000.0))/((4.0090554886458e-26 + 1/(-f**2 - complex(0,20000000000.0)*f + 1.205604e+26))**0.5*(-8.29459756271545e-27*f**2 - complex(0,1.65891951254309e-16)*f + 1)**2)
    return c/(n(f)+ f* np.real(droote))

def vph(f):
    return c/n(f)

######################################################################################################
# GaP EO coefficent (r41)
######################################################################################################
#From Casablbouni PRSTAB 2008
dE = 1e-12 #(m/V)
C0 = -0.53

def r41(f):
    return dE * (1+ ((C0 * fr0**2)/(fr0**2 - f**2 - complex(0,Del0)*f)))

######################################################################################################
# GaP Response function for delta function pulse (laser) 
######################################################################################################
def Gd(f,d,vg_laser_eff):
    a = -complex(0,c)* (-1+np.exp(2*d*np.pi*f*(complex(0,1)*((-1/vg_laser_eff)+(1/vph(f))) -k(f)/c)))
    b = vg_laser_eff *vph(f)/(2*np.pi*d*f*(-c*vph(f) + vg_laser_eff*(c+complex(0,1)*k(f)*vph(f))))
    return a*b

######################################################################################################
# GaP Response function for finite width pulse (laser) 
######################################################################################################

def G(f,d,vg_laser_eff,sig_eff_laser):
    a = complex(0,c)* np.exp(-2 *f*np.pi *( f*np.pi * sig_eff_laser**2 + complex(0,1/vg_laser_eff)*d + d*k(f)/c))
    b = np.exp(2*d*np.pi*f*(complex(0,1/vg_laser_eff))+k(f)/c) - np.exp(complex(0,2)*d*f*np.pi/vph(f))
    d = vg_laser_eff * vph(f) / (2*d*np.pi*f * (-c*vph(f)+vg_laser_eff*(c+complex(0,1)*k(f)*vph(f))))
    return a*b*d

def Gd_interp(f,Gd_array):
    # This bridges over zero (undefined)
    nan_mask = np.isnan(Gd_array)

    if np.any(nan_mask):
        Gd_array[np.where(nan_mask)] = np.interp(
            f[np.where(nan_mask)],
            f[~nan_mask],
            Gd_array[~nan_mask].real
        ) + 1j * np.interp(
            f[np.where(nan_mask)],
            f[~nan_mask],
        Gd_array[~nan_mask].imag
        )
    return Gd_array


######################################################################################################
# Transmission Coefficient
######################################################################################################

def Atr(f):
    return 2/ (1+ n(f)+complex(0,1)*k(f))


######################################################################################################
# Geoemtric response function for delta pulse with transmisstion coefficient
######################################################################################################

def GEOd(f,Gd_array):
    return Gd_array*Atr(f)*r41(f) 


######################################################################################################
#  Reflection Coefficient
######################################################################################################

def Aref(f):
    return (1-n(f) -complex(0,1)*k(f))/(1+n(f) + complex(0,1)*k(f))

######################################################################################################
# Geoemtric response function for delta pulse with reflection coefficient
######################################################################################################

def GEO_ref(f,d):
    # should maybe also use finite pulse but should not make that much of a difference
    return (Aref(f))**2 * np.exp(complex(0,1)* 2* np.pi* f* n(f)*2*d/c)* np.exp(- 2*np.pi*f*k(f)*2*d/c)

######################################################################################################
# Phase Retardation of Laser as function of location on crystal (alpha) with φ(α)
######################################################################################################
def φ1(alpha):
    return (1/2) * np.arccos(np.sin(alpha)/np.sqrt(1+ 3 *np.cos(alpha)**2))
    # return 1

def φ2(alpha):
    alpha = alpha + np.pi/2
    return (1/2) * np.arccos(np.sin(alpha)/np.sqrt(1+ 3 *np.cos(alpha)**2)) # Steffan's Thesis
    # return 1


def Gamma_1(alpha, l_laser,dcry, tot_field):
    Rlaser = (np.pi/l_laser) * (nopt(l_laser))**3 *dcry * np.sqrt(1+ 3 *np.cos(alpha)**2)
    return Rlaser * (tot_field)

def Gamma_2(alpha, l_laser,dcry, tot_field):
    # Rlaser = (np.pi/l_laser) * (nopt(l_laser))**3 *dcry * np.sqrt(1+ 3 *np.sin(alpha)**2)
    alpha = alpha + np.pi/2
    Rlaser = (np.pi/l_laser) * (nopt(l_laser))**3 *dcry * np.sqrt(1+ 3 *np.cos(alpha)**2)
    return Rlaser * (tot_field)

######################################################################################################
# Time window calculation
######################################################################################################

def time_tilt(Rlaser, ax_angle = 10):
    θ1 = np.radians(ax_angle) # axicon angle 
    ng =  1.4671
    n = 1.4533
    θ3 = (θ1*(n-1)) # convergence angle
    θ5 = np.pi/2 - (θ1*(n-1) + np.arctan((1- (θ1**2 * ng*(n-1)))/(θ1*(ng-1))))
    # Rlaser =  Ro - Ri
    c = 3e8
    Tw = Rlaser * (θ5 + θ1*(n-1))/c
    return Tw

######################################################################################################
# Simulation with pulse front tilt calculation
######################################################################################################

def EOS_sim_2b_finite(r01x, r01y, r02x, r02y, time_delay,
                      Q1=1000e-12, Q2=800e-12,
                      sig_1=15e-6/3e8, sig_2=15e-6/3e8, delt12=400e-15,
                      dcry=100e-6, theta=5,ax_angle = 10,
                      pixelsx=2856, pixelsy=2856,
                      Ri=5.2, Ro=13.2, laser=800e-9, sig_eff_laser=50e-15,
                      cryst_geo=True,
                      # --- geometry params ---
                      crystal_size=10e-3,
                      inner_gap_lr=2e-3,   # gap between LEFT and RIGHT crystals
                      inner_gap_tb=2e-3,   # gap between TOP and BOTTOM crystals
                      # --- delay params ---
                      delay_topbottom=True, delay_amount=200e-15,
                      # optional edge masking
                      edges=False):
    """
    Same calculation as old version, but with:
      - 4 rectangular "crystals"
      - left/right: NOT rotated, NOT delayed
      - top/bottom: rotated (use φ2 + Gamma_2), additionally delayed
      - overlaps: contributions ADD (so overlap regions contain both signals)
      - independent gaps: inner_gap_lr and inner_gap_tb
    """

    # ----------------------------
    # meshgrid on crystal
    # ----------------------------
    crystx = Ro * 1e-3
    crysty = Ro * 1e-3

    x = np.linspace(-crystx, crystx, pixelsx)
    y = np.linspace(-crysty, crysty, pixelsy)
    X, Y = np.meshgrid(x, y, sparse=False)
    Rl = np.sqrt(X**2 + Y**2)
    print(f"pixel_size x: {x[1]-x[0]}")
    print(f"pixel_size y: {y[1]-y[0]}")
    # ----------------------------
    # pulse-front tilt time map (unchanged)
    # ----------------------------
    Ric = Ri * 1e-3
    theta_rad = np.radians(theta)

    θ1 = np.radians(ax_angle)
    ng = 1.4671
    n_ = 1.4533
    θ5 = np.pi/2 - (θ1*(n_-1) + np.arctan((1 - (θ1**2 * ng*(n_-1))) / (θ1*(ng-1))))

    Rlaser = (Rl - Ric)
    t = Rlaser * (θ5 + θ1*(n_-1)) / c

    # ----------------------------
    # angles / ratios (unchanged)
    # ----------------------------
    alphab1 = np.mod(np.arctan2(Y - r01y, X - r01x), 2*np.pi)
    alphab2 = np.mod(np.arctan2(Y - r02y, X - r02x), 2*np.pi)

    Rlo1 = np.sqrt((X - r01x)**2 + (Y - r01y)**2)
    Rlo2 = np.sqrt((X - r02x)**2 + (Y - r02y)**2)

    ratio1 = Rlo1 / Rl
    ratio2 = Rlo2 / Rl
    ratio1 = np.nan_to_num(ratio1, nan=0.0, posinf=0.0, neginf=0.0)
    ratio2 = np.nan_to_num(ratio2, nan=0.0, posinf=0.0, neginf=0.0)

    # ---------------------------------------------------------
    # helper: compute total_field1,total_field2 for a time_delay
    # using EXACT SAME pipeline you had before
    # ---------------------------------------------------------
    def compute_total_fields(time_delay_eff):
        f1, Erf1, dt = FThz1d(t, Er1t(Rl, t, time_delay_eff, sig_1, Q1))
        f2, Erf2, dt = FThz1d(t, Er2t(Rl, t, time_delay_eff, delt12, sig_2, Q2))

        Gd1 = Gd_interp(f1, G(f1, dcry, vg_opt_eff(laser, theta_rad), sig_eff_laser))
        Gd2 = Gd_interp(f2, G(f2, dcry, vg_opt_eff(laser, theta_rad), sig_eff_laser))

        GEOd1 = GEOd(f1, Gd1)
        GEOd2 = GEOd(f2, Gd2)

        GEOdref1 = GEO_ref(f1, dcry)
        GEOdref2 = GEO_ref(f2, dcry)

        FEeff1 = Erf1 * GEOd1
        FEeff2 = Erf2 * GEOd2

        FEreff1 = GEOdref1 * FEeff1
        FEreff2 = GEOdref2 * FEeff2

        trans_field1 = ifft_response(f1, FEeff1, dt)
        trans_field2 = ifft_response(f2, FEeff2, dt)

        ref_field1 = ifft_response(f1, FEreff1, dt)
        ref_field2 = ifft_response(f2, FEreff2, dt)

        trans_field1f = recast_1d_to_2d(Rl, trans_field1) * (1/ratio1)
        trans_field2f = recast_1d_to_2d(Rl, trans_field2) * (1/ratio2)

        ref_fieldf1 = recast_1d_to_2d(Rl, ref_field1) * (1/ratio1)
        ref_fieldf2 = recast_1d_to_2d(Rl, ref_field2) * (1/ratio2)

        total_field1 = trans_field1f + ref_fieldf1
        total_field2 = trans_field2f + ref_fieldf2

        total_field1[t < 0] = 0
        total_field2[t < 0] = 0

        total_field1 = np.nan_to_num(total_field1, nan=0.0, posinf=0.0, neginf=0.0)
        total_field2 = np.nan_to_num(total_field2, nan=0.0, posinf=0.0, neginf=0.0)
        return total_field1, total_field2

    # nominal fields (left/right)
    total_field1_nom, total_field2_nom = compute_total_fields(time_delay)

    # delayed fields (top/bottom)
    if delay_topbottom and delay_amount != 0:
        total_field1_del, total_field2_del = compute_total_fields(time_delay + delay_amount)
    else:
        total_field1_del, total_field2_del = total_field1_nom, total_field2_nom

    # ----------------------------
    # build 4-rectangle geometry with TWO gaps
    # ----------------------------
    half = crystal_size / 2

    # independent center offsets for vertical vs horizontal pair
    Ri_m_x = half + inner_gap_lr / 2   # LEFT/RIGHT shift in X
    Ri_m_y = half + inner_gap_tb / 2   # TOP/BOTTOM shift in Y

    m_top   = (np.abs(X) <= half) & (np.abs(Y - Ri_m_y) <= half)
    m_bot   = (np.abs(X) <= half) & (np.abs(Y + Ri_m_y) <= half)
    m_left  = (np.abs(X + Ri_m_x) <= half) & (np.abs(Y) <= half)
    m_right = (np.abs(X - Ri_m_x) <= half) & (np.abs(Y) <= half)

    m_union = m_top | m_bot | m_left | m_right

    # plot geometry (your helper)
    plot_overlap_geometry(x, y, m_top, m_bot, m_left, m_right,
                          title="Crystal Geometry (TB rotated+delayed; LR normal)")

    # ----------------------------
    # compose Intensity/Gamma EXACTLY like old style,
    # but ADD contributions so overlaps contain both signals
    # ----------------------------
    Gamma_map = np.zeros_like(Rl)
    Intensity = np.zeros_like(Rl)

    # helper to ADD contribution for one region
    def add_region(mask, phi_fn, gamma_fn, a1, a2, f1, f2):
        # bunch contributions (keep separate so we can apply phi weighting like old code)
        g1 = gamma_fn(a1[mask], laser, dcry, f1[mask])
        g2 = gamma_fn(a2[mask], laser, dcry, f2[mask])

        # gamma display map: add (so overlaps show larger total Γ)
        Gamma_map[mask] += (g1 + g2)
        # Gamma_map[t<0] = 0

        # intensity: old code form, add the two bunch intensities with phi weighting
        Intensity[mask] += (np.sin(phi_fn(a1[mask]))**2) #* (np.sin(g1/2)**2)
        Intensity[mask] += (np.sin(phi_fn(a2[mask]))**2) #* (np.sin(g2/2)**2)

    # IMPORTANT: order doesn't matter anymore because we ADD.
    # But if you want LR "first", this is LR then TB.
    add_region(m_left,  φ1, Gamma_1, alphab1, alphab2, total_field1_nom, total_field2_nom)
    add_region(m_right, φ1, Gamma_1, alphab1, alphab2, total_field1_nom, total_field2_nom)

    add_region(m_top,   φ2, Gamma_2, alphab1, alphab2, total_field1_del, total_field2_del)
    add_region(m_bot,   φ2, Gamma_2, alphab1, alphab2, total_field1_del, total_field2_del)

    # outside crystals -> 0
    Gamma_map[~m_union] = 0.0
    Intensity[~m_union] = 0.0

    # ----------------------------
    # plots (unchanged)
    # ----------------------------
    fig, axes = plt.subplots(1, 2, figsize=(15, 5), constrained_layout=True)

    im0 = axes[0].imshow(Intensity*100, cmap='plasma', origin='lower',
                         extent=[x.min()*1e3, x.max()*1e3, y.min()*1e3, y.max()*1e3],
                         aspect='equal')
    axes[0].set_title('Intensity')
    axes[0].set_xlabel('x (mm)')
    axes[0].set_ylabel('y (mm)')
    plt.colorbar(im0, ax=axes[0], fraction=0.046, pad=0.04, label='% Transmission at Detector')

    im1 = axes[1].imshow(Gamma_map*1e3, cmap='plasma', origin='lower',
                         extent=[x.min()*1e3, x.max()*1e3, y.min()*1e3, y.max()*1e3],
                         aspect='equal')
    axes[1].set_title(r'Phase Shift (Γ)')
    axes[1].set_xlabel('x (mm)')
    axes[1].set_ylabel('y (mm)')
    plt.colorbar(im1, ax=axes[1], label='mrad', fraction=0.046, pad=0.04)

    plt.show()
    print(f"Maximum Intensity {np.max(Intensity)*100:.3f} %")

    del X, Y, x, y, alphab1, alphab2
    gc.collect()

    return Intensity, Gamma_map, t

In [ ]:
def EOS_sim_2b_finite(r01x, r01y, r02x, r02y, time_delay,
                      Q1=1000e-12, Q2=800e-12,
                      sig_1=15e-6/3e8, sig_2=15e-6/3e8, delt12=400e-15,
                      dcry=100e-6, ax_angle = 10,
                      pixelsx=2856, pixelsy=2856,
                      Ri=5.2, Ro=13.2, laser=800e-9, sig_eff_laser=50e-15,
                      # --- geometry params ---
                      crystal_size=10e-3,
                      inner_gap_lr=2e-3,   # gap between LEFT and RIGHT crystals
                      inner_gap_tb=2e-3,   # gap between TOP and BOTTOM crystals
                      # --- delay params ---
                      delay_topbottom=True, delay_amount=200e-15):
    """
    This calculates the EOS response from GaP 
    - Top/Bottom crystals can be delayed in time with respect to Left/Right to model crystal mount
    - Top/Bottom and Left/Right crystals can be placed at different distances to the ebeam axis of propagation
    - This is calculated at the crystal (the magnification onto the camera is not taken into account here to simplify)

    Inputs: 
    - time_delay: initial time delay of the signal (s)
    - Q1/Q2: charge of the drive and witness bunch respectively (C)
    - r01x, r01y: x and y position of the drive bunch with respect to the center of the crystals 
    - r02x, r02y: x and y position of the witness bunch with respect to the center of the crystals 
    - sig_1/sig_2: FWHM of drive/witness bunch respectively (s)
    - delt12: bunch separation between the bunches (s)
    - dcry: thickness of GaP crystal 
    - ax_angle: axicon angle (not angle to optical axis!)
    - Ri/Ro: Inner and outer radius of the donut of the laser (mm)
    - laser: central wavelength of the laser (m)
    - pixelsx/y: define number of pixels (typically from camera)
    - sig_eff_laser: FWHM of laser pulse (s)
    - crystal size: size of crystal pieces (m)
    - inner_gap_lr/tp: gap between the crystals (m)
    - delay_top_bottom: True (allows to offset crystals in z)
    - delay_amount: gap between crystals in z (s)

    """

    # ----------------------------
    # meshgrid on crystal
    # ----------------------------
    crystx = Ro * 1e-3
    crysty = Ro * 1e-3

    x = np.linspace(-crystx, crystx, pixelsx)
    y = np.linspace(-crysty, crysty, pixelsy)
    X, Y = np.meshgrid(x, y, sparse=False)
    Rl = np.sqrt(X**2 + Y**2)
    print(f"pixel_size x: {x[1]-x[0]}")
    print(f"pixel_size y: {y[1]-y[0]}")

    # ----------------------------
    # pulse-front tilt time map 
    # ----------------------------

    Ric = Ri * 1e-3
    θ1 = np.radians(ax_angle)
    ng = 1.4671
    n_ = 1.4533
    θ5 = np.pi/2 - (θ1*(n_-1) + np.arctan((1 - (θ1**2 * ng*(n_-1))) / (θ1*(ng-1))))
    Rlaser = (Rl - Ric)
    t = Rlaser * (θ5 + θ1*(n_-1)) / c

    # ----------------------------
    # angles / ratios (unchanged)
    # ----------------------------

    alphab1 = np.mod(np.arctan2(Y - r01y, X - r01x), 2*np.pi)
    alphab2 = np.mod(np.arctan2(Y - r02y, X - r02x), 2*np.pi)

    Rlo1 = np.sqrt((X - r01x)**2 + (Y - r01y)**2)
    Rlo2 = np.sqrt((X - r02x)**2 + (Y - r02y)**2)

    ratio1 = Rlo1 / Rl #+ 50e-6
    ratio2 = Rlo2 / Rl #+ 50e-6
    ratio1 = np.nan_to_num(ratio1, nan=0.0, posinf=0.0, neginf=0.0)
    ratio2 = np.nan_to_num(ratio2, nan=0.0, posinf=0.0, neginf=0.0)

    # ---------------------------------------------------------
    # Compute total_field1,total_field2 for a time_delay
    # ---------------------------------------------------------
    
    def compute_total_fields(time_delay_base, t_shift=0.0):
        # Shift the crystal time axis 
        t_use = t - t_shift

        f1, Erf1, dt = FThz1d(t_use, Er1t(Rl, t_use, time_delay_base, sig_1, Q1))
        f2, Erf2, dt = FThz1d(t_use, Er2t(Rl, t_use, time_delay_base, delt12, sig_2, Q2))

        Gd1 = Gd_interp(f1, G(f1, dcry, vg_opt_eff(laser, θ5), sig_eff_laser))
        Gd2 = Gd_interp(f2, G(f2, dcry, vg_opt_eff(laser, θ5), sig_eff_laser))

        GEOd1 = GEOd(f1, Gd1)
        GEOd2 = GEOd(f2, Gd2)

        GEOdref1 = GEO_ref(f1, dcry)
        GEOdref2 = GEO_ref(f2, dcry)

        FEeff1 = Erf1 * GEOd1
        FEeff2 = Erf2 * GEOd2

        FEreff1 = GEOdref1 * FEeff1
        FEreff2 = GEOdref2 * FEeff2

        trans_field1 = ifft_response(f1, FEeff1, dt)
        trans_field2 = ifft_response(f2, FEeff2, dt)

        ref_field1 = ifft_response(f1, FEreff1, dt)
        ref_field2 = ifft_response(f2, FEreff2, dt)

        trans_field1f = recast_1d_to_2d(Rl, trans_field1) * (1/ratio1)
        trans_field2f = recast_1d_to_2d(Rl, trans_field2) * (1/ratio2)

        ref_fieldf1 = recast_1d_to_2d(Rl, ref_field1) * (1/ratio1)
        ref_fieldf2 = recast_1d_to_2d(Rl, ref_field2) * (1/ratio2)

        total_field1 = trans_field1f + ref_fieldf1
        total_field2 = trans_field2f + ref_fieldf2

        total_field1 = np.nan_to_num(total_field1, nan=0.0, posinf=0.0, neginf=0.0)
        total_field2 = np.nan_to_num(total_field2, nan=0.0, posinf=0.0, neginf=0.0)

        return total_field1, total_field2

    total_field1_nom, total_field2_nom = compute_total_fields(time_delay, t_shift=0.0)
    
    if delay_topbottom and delay_amount != 0:
        total_field1_del, total_field2_del = compute_total_fields(time_delay, t_shift=delay_amount)
    else:
        total_field1_del, total_field2_del = total_field1_nom, total_field2_nom

    # ----------------------------
    # build 4-rectangle geometry with TWO gaps
    # ----------------------------
    half = crystal_size / 2

    # independent center offsets for vertical vs horizontal pair
    Ri_m_x = half + inner_gap_lr / 2   # LEFT/RIGHT shift in X
    Ri_m_y = half + inner_gap_tb / 2   # TOP/BOTTOM shift in Y

    m_top   = (np.abs(X) <= half) & (np.abs(Y - Ri_m_y) <= half)
    m_bot   = (np.abs(X) <= half) & (np.abs(Y + Ri_m_y) <= half)
    m_left  = (np.abs(X + Ri_m_x) <= half) & (np.abs(Y) <= half)
    m_right = (np.abs(X - Ri_m_x) <= half) & (np.abs(Y) <= half)

    m_union = m_top | m_bot | m_left | m_right

    # ----------------------------
    # Gamma and Intensity
    # ----------------------------
    Gamma_map = np.zeros_like(Rl)
    Intensity = np.zeros_like(Rl)

    def add_region(mask, phi_fn, gamma_fn, a1, a2, f1, f2):
        # bunch contributions 
        g1 = gamma_fn(a1[mask], laser, dcry, f1[mask])
        g2 = gamma_fn(a2[mask], laser, dcry, f2[mask])
        p1 = phi_fn(a1[mask])
        p2 = phi_fn(a2[mask])
        # gamma display map: add (so overlaps show larger total Γ)

        Gamma_map[mask] += (g1 + g2)

        # Intensity from Gamma and Phi term 
        Intensity[mask] +=  (np.sin((g1)/2)**2) * (np.sin((p1)*2)**2) 
        Intensity[mask] +=  (np.sin((g2)/2)**2) * (np.sin((p2)*2)**2) 

    # creating the 

    add_region(m_left,  φ1, Gamma_1, alphab1, alphab2, total_field1_nom, total_field2_nom)
    add_region(m_right, φ1, Gamma_1, alphab1, alphab2, total_field1_nom, total_field2_nom)

    add_region(m_top,   φ2, Gamma_2, alphab1, alphab2, total_field1_del, total_field2_del)
    add_region(m_bot,   φ2, Gamma_2, alphab1, alphab2, total_field1_del, total_field2_del)

    # outside crystals -> 0
    Gamma_map[~m_union] = 0.0
    Intensity[~m_union] = 0.0
    print("min/max Gamma_map", np.nanmin(Gamma_map), np.nanmax(Gamma_map))

    # ----------------------------
    # plots (unchanged)
    # ----------------------------
    fig, axes = plt.subplots(1, 2, figsize=(15, 5), constrained_layout=True)

    im0 = axes[0].imshow(Intensity*100, cmap='plasma', origin='lower',
                         extent=[x.min()*1e3, x.max()*1e3, y.min()*1e3, y.max()*1e3],
                         aspect='equal')
    axes[0].set_title('Intensity')
    axes[0].set_xlabel('x (mm)')
    axes[0].set_ylabel('y (mm)')
    plt.colorbar(im0, ax=axes[0], fraction=0.046, pad=0.04, label='% Transmission at Detector')

    im1 = axes[1].imshow(Gamma_map*1e3, cmap='plasma', origin='lower',
                         extent=[x.min()*1e3, x.max()*1e3, y.min()*1e3, y.max()*1e3],
                         aspect='equal')
    axes[1].set_title(r'Phase Shift (Γ)')
    axes[1].set_xlabel('x (mm)')
    axes[1].set_ylabel('y (mm)')
    plt.colorbar(im1, ax=axes[1], label='mrad', fraction=0.046, pad=0.04)

    plt.show()
    print(f"Maximum Intensity {np.max(Intensity)*100:.3f} %")


    del X, Y, x, y, alphab1, alphab2
    gc.collect()

    return Intensity, Gamma_map, t

#### Running Simulation

In [ ]:
r01x = 2e-3 #drive
r01y =0e-3 #drive
r02x = 0e-3 #witness
r02y = 0e-3 #witness
time_delay =0e-13 
sig_z2_thz = 40e-6 # length (m)
sig_t2_thz = sig_z2_thz/c 

Intensity, Gamma, t = EOS_sim_2b_finite(r01x, r01y, r02x, r02y, time_delay,
                      Q1=100e-12, Q2= 1000e-12,
                      sig_1=30e-6/3e8, sig_2=30e-6/3e8, delt12=90e-6/c,
                      dcry=100e-6, ax_angle = 10,
                      pixelsx=2464, pixelsy=2056,
                      Ri=5.2, Ro=11.5, laser=800e-9, sig_eff_laser=30e-15,
                      # --- geometry params ---
                      crystal_size=10e-3,
                      inner_gap_lr=3e-3*2,   # gap between LEFT and RIGHT crystals
                      inner_gap_tb=3e-3*2,   # gap between TOP and BOTTOM crystals
                      # --- delay params ---
                      delay_topbottom=True, delay_amount=50e-6/c)

In [ ]:
0.2* 600* 0.145

In [ ]:
1e-12 * 1e9

In [ ]:
plt.plot(Intensity[2464//2])
plt.xlim(1600,2464)

In [ ]:
r01x = -0e-3 #drive
r01y =0e-3 #drive
r02x = 0e-3 #witness
r02y = 0e-3 #witness
time_delay =4e-13
sig_z2_thz = 40e-6 # length (m)
sig_t2_thz = sig_z2_thz/c 

# Intensity, Gamma, t = EOS_sim_2b_finite(r01x, r01y, r02x, r02y, time_delay,
#                       Q1=800e-12, Q2=800e-12,
#                       sig_1=30e-6/3e8, sig_2=30e-6/3e8, delt12=300e-15,
#                       dcry=100e-6, theta=5,ax_angle = 10,
#                       pixelsx=2464, pixelsy=2056,
#                       Ri=3.4, Ro=13.2, laser=800e-9, sig_eff_laser=50e-15,
#                       cryst_geo=True,
#                       # --- geometry params ---
#                       crystal_size=10e-3,
#                       inner_gap_lr=3.4e-3*2,   # gap between LEFT and RIGHT crystals
#                       inner_gap_tb=3.4e-3*2,   # gap between TOP and BOTTOM crystals
#                       # --- delay params ---
#                       delay_topbottom=True, delay_amount=100e-6/c,
#                       # optional edge masking
#                       edges=False)

Intensity, Gamma, t = EOS_sim_2b_finite(r01x, r01y, r02x, r02y, time_delay,
                      Q1=1e-9, Q2=100e-12,
                      sig_1=30e-6/3e8, sig_2=30e-6/3e8, delt12=400e-15,
                      dcry=100e-6, ax_angle = 7.8,
                      pixelsx=2464, pixelsy=2056,
                      Ri=4.61, Ro=11.35, laser=800e-9, sig_eff_laser=50e-15,
                      # --- geometry params ---
                      crystal_size=10e-3,
                      inner_gap_lr=5e-3*2,   # gap between LEFT and RIGHT crystals
                      inner_gap_tb=5e-3*2,   # gap between TOP and BOTTOM crystals
                      # --- delay params ---
                      delay_topbottom=True, delay_amount=0e-6/c)

#### Testing Time window

In [ ]:
# Constants
Ro_ax = 1.6e-2 / 2
delta = 3.5e-2

# Distance range
d = np.linspace(0.1, 0.33, 2000)

# Fixed mirror radius
Ri_m = 2.23e-3

# Sweep crystal radii
Ri_c_vals = np.linspace(3e-3, 6e-3, 100)

theta_solutions = []
d_solutions = []

for Ri_c in Ri_c_vals:
    theta_c = np.degrees(np.arctan((Ri_c + Ro_ax)/d)) * 2
    theta_m = np.degrees(np.arctan((Ri_m + Ro_ax)/(d - delta))) * 2
    
    diff = np.abs(theta_c - theta_m)
    idx = np.argmin(diff)
    
    theta_solutions.append(theta_c[idx])
    d_solutions.append(d[idx])

theta_solutions = np.array(theta_solutions)
d_solutions = np.array(d_solutions)

#### Scan over Radius and time

In [ ]:
import numpy as np

# Define time delays (example)
time_delays = np.linspace(0e-13, 5e-13, 20) + 2*30e-6/3e8

# Storage arrays
Int_max_2D = np.zeros((len(time_delays), len(Ri_c_vals)))

for j, td in enumerate(time_delays):
    for i in range(len(Ri_c_vals)):

        Intensity, Gamma, t = EOS_sim_2b_finite(
            r01x, r01y, r02x, r02y, td,
            Q1=800e-9, Q2=800e-12,
            sig_1=30e-6/3e8, sig_2=30e-6/3e8, delt12=135e-6/c,
            dcry=100e-6, theta=5, ax_angle=theta_solutions[i],
            pixelsx=2464, pixelsy=2056,
            Ri=Ri_c_vals[i]*1e3, Ro=11.35,
            laser=800e-9, sig_eff_laser=50e-15,
            cryst_geo=True,
            crystal_size=10e-3,
            inner_gap_lr=6e-3,
            inner_gap_tb=10e-3,
            delay_topbottom=True,
            delay_amount=100e-6/c,
            edges=False
        )

        Int_max_2D[j, i] = np.max(Intensity)

In [ ]:
plt.figure(figsize=(7, 5))

plt.imshow(
    Int_max_2D*100,
    aspect='auto',
    origin='lower',
    extent=[
        theta_solutions.min(), theta_solutions.max(),
        time_delays.min()*1e15+135e-6/c*1e15, time_delays.max()*1e15+135e-6/c*1e15
        # time_delays.min()*1e15, time_delays.max()*1e15
    ],
    cmap='inferno'
)


plt.xlabel("Axicon Angle (deg)")
plt.ylabel("Time Delay (fs)")
plt.title("Max Intensity vs Angle & Time Delay")
plt.colorbar(label="Max Intensity")

plt.show()

In [ ]:
target_time = 100e-15  # 100 fs in seconds

idx = np.argmin(np.abs(time_delays - target_time))

plt.imshow(
    Int_max_2D*100,
    aspect='auto',
    origin='lower',
    extent=[
        Ri_c_vals.min()*1e3, Ri_c_vals.max()*1e3,
        # time_delays.min()*1e15, time_delays.max()*1e15
        time_delays.min()*1e15+135e-6/c*1e15, time_delays.max()*1e15+135e-6/c*1e15
    ],
    cmap='inferno'
)

plt.xlabel("Inner Radius (mm)")
plt.ylabel("Time Delay (fs)")
plt.title("Max Intensity vs Inner Radius & Time Delay")
plt.colorbar(label="Max Intensity")
# plt.ylim(100,500)

plt.show()

In [ ]:
idx_flat = np.argmax(Int_max_2D)
j, i = np.unravel_index(idx_flat, Int_max_2D.shape)
max_intensity = Int_max_2D[j, i]

best_theta = theta_solutions[i]
best_Ri = Ri_c_vals[i]
best_time = time_delays[j]

print(f"Max Intensity: {max_intensity}")
print(f"Axicon Angle: {best_theta:.3f} deg")
print(f"Inner Radius: {best_Ri*1e3:.2f} mm")
print(f"Time Delay: {best_time*1e15:.2f} fs")

In [ ]:
idx_flat = np.argmax(Int_max_2D)
j, i = np.unravel_index(idx_flat, Int_max_2D.shape)
max_intensity = Int_max_2D[j, i]

best_theta = theta_solutions[i]
best_Ri = Ri_c_vals[i]
best_time = time_delays[j]

print(f"Max Intensity: {max_intensity}")
print(f"Axicon Angle: {best_theta:.3f} deg")
print(f"Inner Radius: {best_Ri*1e3:.2f} mm")
print(f"Time Delay: {best_time*1e15:.2f} fs")

In [ ]:
Intensity, Gamma, t = EOS_sim_2b_finite(r01x, r01y, r02x, r02y,0e-15+2*30e-6/3e8+135e-6/c,
                    Q1=0, Q2=100e-12,
                    sig_1=30e-6/3e8, sig_2=30e-6/3e8, delt12=0,
                    dcry=100e-6, theta=5,ax_angle = best_theta,
                    pixelsx=2464, pixelsy=2056,
                    Ri=best_Ri*1e3, Ro=11.35, laser=800e-9, sig_eff_laser=50e-15,
                    cryst_geo=True,
                    # --- geometry params ---
                    crystal_size=10e-3,
                    inner_gap_lr=6e-3,   # gap between LEFT and RIGHT crystals
                    inner_gap_tb=10e-3,   # gap between TOP and BOTTOM crystals
                    # --- delay params ---
                    delay_topbottom=True, delay_amount=0e-6/c,
                    # optional edge masking
                    edges=False)

In [ ]:
Intensity, Gamma, t = EOS_sim_2b_finite(r01x, r01y, r02x, r02y,0e-15+2*30e-6/3e8,
                    Q1=1.6e-9, Q2=0e-12,
                    sig_1=30e-6/3e8, sig_2=30e-6/3e8, delt12=200e-15,
                    dcry=100e-6, theta=5,ax_angle = 10,
                    pixelsx=2464, pixelsy=2056,
                    Ri=5.2, Ro=11.35, laser=800e-9, sig_eff_laser=50e-15,
                    cryst_geo=True,
                    # --- geometry params ---
                    crystal_size=10e-3,
                    inner_gap_lr=6e-3,   # gap between LEFT and RIGHT crystals
                    inner_gap_tb=10e-3,   # gap between TOP and BOTTOM crystals
                    # --- delay params ---
                    delay_topbottom=True, delay_amount=0e-6/c,
                    # optional edge masking
                    edges=False)

##### Saving Data

In [ ]:
path = path = "outputs/Ax_optimization/"
np.savez("full_scan_results_no_ref_w.npz",
         Int_max_2D=Int_max_2D,
         time_delays=time_delays,
         Ri_c_vals=Ri_c_vals,
         theta_solutions=theta_solutions,
         d_solutions=d_solutions)

##### Time resolution calculation

In [ ]:
pix_t = np.sqrt( 9.216402760860512e-06**2 + 1.1046228710462694e-05**2)
time_res = time_tilt(pix_t, theta_solutions)
plt.plot(theta_solutions,time_res*1e15)
plt.title("TOA Resolution")
plt.xlabel("axicon angle (deg)")
plt.ylabel("fs/pixel")

In [ ]:
pix_t = np.sqrt( 9.216402760860512e-06**2 + 1.1046228710462694e-05**2)
time_res = time_tilt(pix_t, theta_solutions)
plt.plot(Ri_c_vals,time_res*1e15)
plt.title("TOA Resolution")
plt.xlabel("Inner Radius")
plt.ylabel("fs/pixel")

In [ ]:
# Define theta range (broader than your solutions)
theta_vals = np.linspace(min(theta_solutions), max(theta_solutions), 200)

# Create 2D grid
Ri_grid, theta_grid = np.meshgrid(Ri_c_vals, theta_vals)

time_res_2D = time_tilt(Ri_grid,theta_grid)

plt.figure(figsize=(7,5))

plt.imshow(
    time_res_2D * 1e12,
    aspect='auto',
    origin='lower',
    extent=[
        Ri_c_vals.min()*1e3, Ri_c_vals.max()*1e3,
        theta_vals.min(), theta_vals.max()
    ],
    cmap='viridis'
)

plt.xlabel("Inner Radius (mm)")
plt.ylabel("Axicon Angle (deg)")
plt.title("Time Window (ps)")
plt.colorbar(label="ps")
plt.axhline(best_theta)
plt.axvline(best_Ri*1e3)

plt.show()

#### Scan over Radius

In [ ]:
# Constants
Ro_ax = 1.6e-2 / 2
delta = 3.5e-2

# Distance range
d = np.linspace(0.1, 0.33, 2000)

# Fixed mirror radius
Ri_m = 2.23e-3

# Sweep crystal radii
Ri_c_vals = np.linspace(3e-3, 6e-3, 100)

theta_solutions = []
d_solutions = []

for Ri_c in Ri_c_vals:
    theta_c = np.degrees(np.arctan((Ri_c + Ro_ax)/d)) * 2
    theta_m = np.degrees(np.arctan((Ri_m + Ro_ax)/(d - delta))) * 2
    
    diff = np.abs(theta_c - theta_m)
    idx = np.argmin(diff)
    
    theta_solutions.append(theta_c[idx])
    d_solutions.append(d[idx])

theta_solutions = np.array(theta_solutions)
d_solutions = np.array(d_solutions)

In [ ]:
time_delay = 0e-13 + 2*30e-6/3e8
Int_max = []
Int_full = []
Gamma_full = []
for i in range(len(Ri_c_vals)):
    Intensity, Gamma, t = EOS_sim_2b_finite(r01x, r01y, r02x, r02y, time_delay,
                      Q1=800e-12, Q2=800e-12,
                      sig_1=30e-6/3e8, sig_2=30e-6/3e8, delt12=100e-6/c,
                      dcry=100e-6, theta=5,ax_angle = theta_solutions[i],
                      pixelsx=2464, pixelsy=2056,
                      Ri=Ri_c_vals[i]*1e3, Ro=11.35, laser=800e-9, sig_eff_laser=50e-15,
                      cryst_geo=True,
                      # --- geometry params ---
                      crystal_size=10e-3,
                      inner_gap_lr=6e-3,   # gap between LEFT and RIGHT crystals
                      inner_gap_tb=6e-3,   # gap between TOP and BOTTOM crystals
                      # --- delay params ---
                      delay_topbottom=True, delay_amount=50e-6/c,
                      # optional edge masking
                      edges=False)
    Gamma_full.append(Gamma)
    Int_full.append(Intensity) 
    Int_max.append(np.max(Intensity))


#### Lineout of Intensity

In [ ]:
# Convert lists to arrays
Int_full = np.array(Int_full)   # shape: (N_theta, Ny, Nx)
theta_vals = np.array(theta_solutions)

# Choose a horizontal slice (e.g., center row)
Ny = Int_full.shape[1]
y_center = Ny // 2

# Extract horizontal lineouts
lineouts = Int_full[:, y_center, :]   # shape: (N_theta, Nx)

# Plot
plt.figure(figsize=(6,4))

extent = [
    0, lineouts.shape[1],              # x: pixels
    theta_vals.min(), theta_vals.max() # y: axicon angle
]

im = plt.imshow(lineouts*100,
                aspect='auto',
                extent=extent,
                origin='lower')

plt.xlabel("Pixel (horizontal)")
plt.ylabel("Axicon angle (theta)")
cbar = plt.colorbar(im)
cbar.set_label("Intensity")

plt.title("Intensity vs Pixel vs Axicon Angle")
plt.tight_layout()
plt.show()

In [ ]:
# Convert lists to arrays
Int_full = np.array(Int_full)   # shape: (N_theta, Ny, Nx)
theta_vals = np.array(theta_solutions)

# Choose a horizontal slice (e.g., center row)
Ny = Int_full.shape[1]
y_center = Ny // 2

# Extract horizontal lineouts
lineouts = Int_full[:, y_center, :]   # shape: (N_theta, Nx)

# Plot
plt.figure(figsize=(6,4))

extent = [
    0, lineouts.shape[1],              # x: pixels
    Ri_c_vals.min()*1e3, Ri_c_vals.max()*1e3 # y: axicon angle
]

im = plt.imshow(lineouts*100,
                aspect='auto',
                extent=extent,
                origin='lower')

plt.xlabel("Pixel (horizontal)")
plt.ylabel("Inner Radius (mm)")
cbar = plt.colorbar(im)
cbar.set_label("Intensity")

plt.title("Intensity vs Pixel vs Inner Radius")
plt.tight_layout()
plt.show()

#### Lineout of Gamma

In [ ]:
# Convert lists to arrays
Gamma = np.array(Gamma_full)   # shape: (N_theta, Ny, Nx)
theta_vals = np.array(theta_solutions)

# Choose a horizontal slice (e.g., center row)
Ny = Int_full.shape[1]
Nx = Int_full.shape[2]
y_center = Ny // 2
x_center = Nx // 2
# Extract horizontal lineouts
lineouts = Gamma[:, y_center, :]   # shape: (N_theta, Nx)
# lineouts = Gamma[:, :, x_center]

# Plot
plt.figure(figsize=(6,4))

extent = [
    0, lineouts.shape[1],              # x: pixels
    theta_vals.min(), theta_vals.max() # y: axicon angle
]

im = plt.imshow(lineouts,
                aspect='auto',
                extent=extent,
                origin='lower')

plt.xlabel("Pixel (horizontal)")
plt.ylabel("Axicon angle (theta)")
cbar = plt.colorbar(im)
cbar.set_label("Intensity")

plt.title("Gamma vs Pixel vs Axicon Angle")
plt.tight_layout()
plt.show()

In [ ]:
# Convert lists to arrays
Gamma = np.array(Gamma_full)   # shape: (N_theta, Ny, Nx)
theta_vals = np.array(theta_solutions)

# Choose a horizontal slice (e.g., center row)
Ny = Int_full.shape[1]
y_center = Ny // 2

# Extract horizontal lineouts
lineouts = Gamma[:, y_center, :]   # shape: (N_theta, Nx)

# Plot
plt.figure(figsize=(6,4))

extent = [
    0, lineouts.shape[1],              # x: pixels
    Ri_c_vals.min()*1e3, Ri_c_vals.max()*1e3 # y: axicon angle
]

im = plt.imshow(lineouts,
                aspect='auto',
                extent=extent,
                origin='lower')

plt.xlabel("Pixel (horizontal)")
plt.ylabel("Inner Radius (mm)")
cbar = plt.colorbar(im)
cbar.set_label("Intensity")

plt.title("Gamma vs Pixel vs Inner Radius")
plt.tight_layout()
plt.show()

In [ ]:
plt.plot(theta_solutions,np.array(Int_max)*100)
plt.title("Axicon Angle vs. Intensity of Signal")
plt.xlabel("Axicon Angle (deg)")
plt.ylabel("Inentisty (%)")
# plt.axvline(7.8)

In [ ]:
plt.plot(Ri_c_vals*1e3,np.array(Int_max)*100)
plt.title("Inner Radius on Crystal vs. Signal Intensity")
plt.xlabel("Ri (mm)")
plt.ylabel("Inentisty (%)")
# plt.axvline(4.62)

In [ ]:
print(f"axicon distance from crystal {d_solutions[np.argmax(Int_max)]*1e3} mm")
print(f"axicon distance from mirror {(d_solutions[np.argmax(Int_max)]-3.5e-2)*1e3} mm")
print(f"Ri on crystal {Ri_c_vals[np.argmax(Int_max)]*1e3}")
print(f"Axicon Angle {theta_solutions[np.argmax(Int_max)]}")

In [ ]:
time_delay = 1e-13
Intensity, Gamma, t = EOS_sim_2b_finite(r01x, r01y, r02x, r02y, time_delay,
                    Q1=1.6e-9, Q2=0e-12,
                    sig_1=30e-6/3e8, sig_2=30e-6/3e8, delt12=400e-15,
                    dcry=100e-6, theta=5,ax_angle = theta_solutions[np.argmax(Int_max)],
                    pixelsx=2464, pixelsy=2056,
                    Ri=Ri_c_vals[np.argmax(Int_max)]*1e3, Ro=11.35, laser=800e-9, sig_eff_laser=50e-15,
                    cryst_geo=True,
                    # --- geometry params ---
                    crystal_size=10e-3,
                    inner_gap_lr=2e-3,   # gap between LEFT and RIGHT crystals
                    inner_gap_tb=10e-3,   # gap between TOP and BOTTOM crystals
                    # --- delay params ---
                    delay_topbottom=True, delay_amount=100e-6/c,
                    # optional edge masking
                    edges=False)

In [ ]:
np.shape(Intensity)

In [ ]:
plt.plot(Intensity[2056//2])

In [ ]:
plt.figure(figsize = (8,7))
for i in range(len(Int_full)):
    plt.plot(Int_full[i][2056//2]*100, label = f"{theta_solutions[i]:.2f}")
plt.xlim(2848//2,2484)
plt.title("Lineout of Intensity")
plt.ylabel("Intensity %" )
plt.xlabel("pixel")
plt.legend()

In [ ]:
plt.imshow(Int_full[0], origin = "lower", cmap = "inferno", aspect = "auto", vmax = np.max(Int_full))
plt.title(f"Intentsity: Ri {Ri_c_vals[0]*1e3} | Ax angle {theta_solutions[0]:.3f}")
plt.colorbar()

In [ ]:
vmin = min(np.min(img) for img in Int_full)
vmax = max(np.max(img) for img in Int_full)

In [ ]:
430*1e-15*3e8 *1e6

#### Plotting and saving

In [ ]:
path = "outputs/Ax_optimization/test_both/"
for i in range(len(Int_full)):
    plt.figure()
    plt.imshow(Int_full[i]*100, origin = "lower", cmap = "PuRd", aspect = "auto",vmax = vmax*100)
    plt.title(f"Intentsity: Ri {Ri_c_vals[i]*1e3} | Ax angle {theta_solutions[i]:.3f}")
    plt.colorbar()
    plt.savefig(path+f"Int_{i}.png")
    plt.close()

In [ ]:
path = "outputs/Ax_optimization/test_drive_lineout/"
for i in range(len(Int_full)):
    Int_f = np.array(Int_full)
    Int = Int_f[i]
    Ny = Int_f.shape[1]
    y_center = Ny // 2
    lineout = Int[y_center, :]
    plt.figure()
    plt.plot(lineout*100)
    plt.ylim(np.min(Int_full)*100,np.max(Int_full)*100)
    plt.title(f"Intentsity: Ri {Ri_c_vals[i]*1e3:.3f} | Ax angle {theta_solutions[i]:.3f}")
    plt.savefig(path+f"Int_{i}.png")
    plt.close()

In [ ]:
path = "outputs/Ax_optimization/test_drive_lineout_g/"
for i in range(len(Int_full)):
    Int_f = np.array(Gamma_full)
    Int = Int_f[i]
    Ny = Int_f.shape[1]
    y_center = Ny // 2
    lineout = Int[y_center, :]
    plt.figure()
    plt.plot(lineout)
    plt.ylim(np.min(Gamma_full),np.max(Gamma_full))
    plt.title(f"Gamma: Ri {Ri_c_vals[i]*1e3:.3f} | Ax angle {theta_solutions[i]:.3f}")
    plt.savefig(path+f"Int_{i}.png")
    plt.close()

In [ ]:
time_tilt((12.45-4.45)*1e-3, 7.31)*1e12

### Testing Geometry

In [ ]:
def test_crystal_geometry(pixelsx=2856, pixelsy=2856,
                          Ro=13.2,               # mm, only used for plotting extent
                          crystal_size=10e-3,    # m (square side length)
                          inner_gap_lr=2e-3,     # m (gap between LEFT/RIGHT)
                          inner_gap_tb=2e-3,     # m (gap between TOP/BOTTOM)
                          show_outlines=True):
    """
    Build and plot ONLY the 4-rectangle cross geometry with two independent gaps.
    Also plots an overlap-count map to show where 2 crystals overlap.
    """

    # ----------------------------
    # meshgrid on "crystal plane"
    # ----------------------------
    crystx = Ro * 1e-3
    crysty = Ro * 1e-3
    x = np.linspace(-crystx, crystx, pixelsx)
    y = np.linspace(-crysty, crysty, pixelsy)
    X, Y = np.meshgrid(x, y, sparse=False)

    # ----------------------------
    # build 4-rectangle geometry with TWO gaps
    # ----------------------------
    half = crystal_size / 2

    # centers are separated by (crystal_size + gap)
    # so each center is offset by half that: half + gap/2
    Ri_m_x = half + inner_gap_lr / 2  # LEFT/RIGHT shift in X
    Ri_m_y = half + inner_gap_tb / 2  # TOP/BOTTOM shift in Y

    m_top   = (np.abs(X) <= half) & (np.abs(Y - Ri_m_y) <= half)
    m_bot   = (np.abs(X) <= half) & (np.abs(Y + Ri_m_y) <= half)
    m_left  = (np.abs(X + Ri_m_x) <= half) & (np.abs(Y) <= half)
    m_right = (np.abs(X - Ri_m_x) <= half) & (np.abs(Y) <= half)

    # overlap count: 0 (none), 1 (single crystal), 2 (overlap of two)
    overlap_count = (m_top.astype(int) + m_bot.astype(int) +
                     m_left.astype(int) + m_right.astype(int))

    # "label" map: (priority only for visualization)
    # 0 none, 1 top, 2 bottom, 3 left, 4 right
    # NOTE: this is NOT physics ordering—just a display choice.
    label = np.zeros_like(overlap_count, dtype=int)
    label[m_top] = 1
    label[m_bot] = 2
    label[m_left] = 3
    label[m_right] = 4

    # ----------------------------
    # plots
    # ----------------------------
    extent = [x.min()*1e3, x.max()*1e3, y.min()*1e3, y.max()*1e3]  # mm

    fig, axes = plt.subplots(1, 2, figsize=(12, 5), constrained_layout=True)

    # ---- left: label map ----
    cmap = ListedColormap(["black", "tab:blue", "tab:green", "tab:orange", "tab:red"])
    norm = BoundaryNorm([-0.5, 0.5, 1.5, 2.5, 3.5, 4.5], cmap.N)

    im0 = axes[0].imshow(label, origin="lower", extent=extent, cmap=cmap, norm=norm, aspect="equal")
    axes[0].set_title("Crystal labels (display only)")
    axes[0].set_xlabel("x (mm)")
    axes[0].set_ylabel("y (mm)")

    # manual colorbar with names
    cbar0 = plt.colorbar(im0, ax=axes[0], fraction=0.046, pad=0.04, ticks=[0,1,2,3,4])
    cbar0.ax.set_yticklabels(["none", "top", "bottom", "left", "right"])

    # optional outlines to see boundaries precisely
    if show_outlines:
        for m, col in [(m_top, "w"), (m_bot, "w"), (m_left, "w"), (m_right, "w")]:
            axes[0].contour(m.astype(float), levels=[0.5], colors=col, linewidths=0.8, origin="lower", extent=extent)

    # ---- right: overlap count map ----
    im1 = axes[1].imshow(overlap_count, origin="lower", extent=extent, aspect="equal")
    axes[1].set_title("Overlap count (0 none, 1 single, 2 overlap)")
    axes[1].set_xlabel("x (mm)")
    axes[1].set_ylabel("y (mm)")
    plt.colorbar(im1, ax=axes[1], fraction=0.046, pad=0.04, label="count")

    if show_outlines:
        # show where overlap_count>=2
        axes[1].contour((overlap_count >= 2).astype(float), levels=[0.5],
                        colors="w", linewidths=1.0, origin="lower", extent=extent)

    plt.show()

    # quick numeric diagnostics
    pix_area = (x[1]-x[0]) * (y[1]-y[0])  # m^2 per pixel
    area_overlap = np.sum(overlap_count >= 2) * pix_area
    area_union = np.sum(overlap_count >= 1) * pix_area

    print(f"center offsets: Ri_m_x={Ri_m_x*1e3:.3f} mm, Ri_m_y={Ri_m_y*1e3:.3f} mm")
    print(f"union area:     {area_union*1e6:.3f} mm^2")
    print(f"overlap area:   {area_overlap*1e6:.3f} mm^2")
    print(f"overlap fraction of union: {(area_overlap/area_union*100 if area_union>0 else 0):.2f} %")

    # return dict(x=x, y=y, X=X, Y=Y,
    #             m_top=m_top, m_bot=m_bot, m_left=m_left, m_right=m_right,
    #             overlap_count=overlap_count, label=label)

In [ ]:
test_crystal_geometry(pixelsx=2856, pixelsy=2856,
                          Ro=13.2,               # mm, only used for plotting extent
                          crystal_size=10.1e-3,    # m (square side length)
                          inner_gap_lr=10e-3,     # m (gap between LEFT/RIGHT)
                          inner_gap_tb=10e-3,     # m (gap between TOP/BOTTOM)
                          show_outlines=True)

### Analysis Code

In [ ]:
def line_cut(image, window):
    '''
    This function takes a lineout of the image in x and y 
    Parameters:
        image (2D np.array): image of EOS BPM data
        window (integer number):  number of rows to sum over for lineout 

    returns: (np.array) lineout x,  lineout y

    '''

    # Image dimensions
    h, w = image.shape
    center_y, center_x = h // 2, w // 2

    # Row range around center_y
    y_start = max(center_y - window // 2, 0)
    y_end = min(center_y + window // 2 + 1, h)

    # Column range around center_x
    x_start = max(center_x - window // 2, 0)
    x_end = min(center_x + window // 2 + 1, w)

    # Horizontal line cut: average across a few rows
    line_x = np.mean(image[y_start:y_end, :], axis=0)

    # Vertical line cut: average across a few columns
    line_y = np.mean(image[:, x_start:x_end], axis=1)

    return line_x, line_y


def peak_loc_sides_bright_inner(
    line,
    prominence=0.05,
    distance=50,
    one_ring_only=False,
):
    """
    Find peaks on both sides of the center.
    - The 'inner' peak on each side is defined as the **highest-intensity** peak.
    - The 'outer' peak is a second peak further from the center than the inner one.
    - If not found, the missing outer (or both) are returned as 0.
    - If one_ring_only=True, only the bright 'inner' is returned; the 'outer' is 0.

    Returns:
        (left_out, left_in, right_in, right_out)  # pixel indices (ints)
    """
    n = len(line)
    center = n // 2

    peaks, _ = find_peaks(line, prominence=prominence * float(np.max(line)), distance=distance)

    left = [p for p in peaks if p < center]
    right = [p for p in peaks if p > center]

    def choose_side(side_peaks, is_left):
        if not side_peaks:
            # no peaks on this side
            return (0, 0) if is_left else (0, 0)

        # inner = brightest on this side
        inner = max(side_peaks, key=lambda p: line[p])

        if one_ring_only:
            # only return the bright ring
            return (0, int(inner)) if is_left else (int(inner), 0)

        # candidates for 'outer' must be farther from the center than the inner one
        if is_left:
            # farther from center means smaller index than inner on the left
            outer_candidates = [p for p in side_peaks if p < inner]
            if not outer_candidates:
                return (0, int(inner))
            outer = max(outer_candidates, key=lambda p: line[p])
            # order: (left_out, left_in)
            return (int(outer), int(inner))
        else:
            # right side: farther from center means larger index than inner
            outer_candidates = [p for p in side_peaks if p > inner]
            if not outer_candidates:
                return (int(inner), 0)
            outer = max(outer_candidates, key=lambda p: line[p])
            # order: (right_in, right_out)
            return (int(inner), int(outer))

    left_out, left_in = choose_side(left, is_left=True)
    right_in, right_out = choose_side(right, is_left=False)

    return (left_out, left_in, right_in, right_out)


def circle(image,window, prominence, distance, peak1 = True):
    
    line_xf ,line_yf= line_cut(image, window)

    left_outfx, left_infx, right_infx, right_outfx = peak_loc_sides_bright_inner(line_xf,prominence, distance)
    left_outfy, left_infy, right_infy, right_outfy = peak_loc_sides_bright_inner(line_yf,prominence, distance)

    x0_cent = (left_infx + right_infx)/2
    y0_cent = (left_infy + right_infy)/2

    x1_rad = np.abs(x0_cent -right_infx) 
    y1_rad = np.abs(y0_cent -left_infy)

    # l_r01 = x1_rad 
    l_r01 = (x1_rad +y1_rad)/2

    x2_rad = np.abs(x0_cent -right_outfx)
    y2_rad = np.abs(y0_cent -left_outfy)

    # l_r02 = x2_rad
    l_r02 = (x2_rad + y2_rad)/2

    return x0_cent, y0_cent, l_r01, l_r02 

def plot_line_cut(image, window, prominence= 0.1, distance = 1):
    # Get lineouts
    line_x, line_y = line_cut(image, window)

    # Detect peaks
    x_left_out, x_left_in, x_right_in, x_right_out = peak_loc_sides_bright_inner(line_x, prominence, distance)
    y_left_out, y_left_in, y_right_in, y_right_out = peak_loc_sides_bright_inner(line_y, prominence, distance)

    # Compute centers and radii
    x0_cent = (x_left_in + x_right_in) / 2
    y0_cent = (y_left_in + y_right_in) / 2

    x_radius = abs(x_right_in - x0_cent)
    x_radius2 = abs(x_right_out - x0_cent)

    y_radius = abs(y_left_in - y0_cent)
    y_radius2 = abs(y_left_out - y0_cent)

    l_r01 = x_radius
    l_r02 = x_radius2

    # === PLOT ===
    plt.figure(figsize=(12, 4))

    # --- X direction ---
    plt.subplot(1, 2, 1)
    plt.plot(line_x, label='X profile')

    # Vertical lines at peaks
    plt.axvline(x_left_in, color='green', linestyle=':', label='Inner peaks')
    plt.axvline(x_right_in, color='green', linestyle=':')
    plt.axvline(x_left_out, color='red', linestyle=':', label='Outer peaks')
    plt.axvline(x_right_out, color='red', linestyle=':')

    # Center
    plt.axvline(x0_cent, color='black', linestyle='--', label='Center')

    # Radii
    plt.axvline(x0_cent + l_r01, color='orange', linestyle='--', label='Radius 1')
    plt.axvline(x0_cent + l_r02, color='blue', linestyle='--', label='Radius 2')

    plt.title('Line Out (X direction)')
    plt.xlabel('Pixel')
    plt.ylabel('Intensity')
    plt.legend()

    # --- Y direction ---
    plt.subplot(1, 2, 2)
    plt.plot(line_y, label='Y profile')

    plt.axvline(y_left_in, color='green', linestyle=':', label='Inner peaks')
    plt.axvline(y_right_in, color='green', linestyle=':')
    plt.axvline(y_left_out, color='red', linestyle=':', label='Outer peaks')
    plt.axvline(y_right_out, color='red', linestyle=':')

    plt.axvline(y0_cent, color='black', linestyle='--', label='Center')

    plt.axvline(y0_cent + l_r01, color='orange', linestyle='--', label='Radius 1')
    plt.axvline(y0_cent + l_r02, color='blue', linestyle='--', label='Radius 2')

    plt.title('Line Out (Y direction)')
    plt.xlabel('Pixel')
    plt.ylabel('Intensity')
    plt.legend()

    plt.tight_layout()
    plt.show()

In [ ]:
image = Intensity
x0_cent, y0_cent, l_r01, l_r02 = circle(image, 20, prominence=0.20, distance = 1, peak1 = True)
print(f"From Line cut")
print(f"x center: {x0_cent}, y center: {y0_cent}")
print(f"inner radius: {l_r01}, outer radius: {l_r02}")
plot_line_cut(image, 20)

In [ ]:
x0_cent, y0_cent = 2464//2,2056//2
pix_x = 1.0718635809987831e-05
pix_y = 1.2846715328466984e-05

# pix_x = 1.0718635809987831e-05
# pix_y = 1.2846715328466984e-05

peak1t = time_tilt(l_r01*pix_x)
peak2t = time_tilt(l_r02*pix_x)

print(f"TOA of peak 1: {peak1t*1e12} ps")
print(f"TOA peak 2: {peak2t*1e12} ps")
print(f"TOA difference: {peak2t*1e12- peak1t*1e12} ps")

#### TOA Analysis

In [ ]:
import numpy as np

def radial_profile_aniso_fast_masked(image, center, pix_x, pix_y,
                                     mask=None,
                                     r_min=0.0, r_max=None, bin_width=None,
                                     mode='sum', threshold=None,
                                     return_counts=False):
    """
    Fast radial profile with anisotropic pixel sizes, optionally masked.

    image: (ny, nx)
    center: (x0, y0) in pixel coords
    pix_x, pix_y: physical pixel sizes (same units as desired r)
    mask: boolean array same shape as image; True pixels are INCLUDED
    """
    x0, y0 = center
    ny, nx = image.shape

    if bin_width is None:
        bin_width = min(pix_x, pix_y)

    # coordinate vectors (no full mesh)
    x = (np.arange(nx) - x0) * pix_x
    y = (np.arange(ny) - y0) * pix_y

    r2 = y[:, None]**2 + x[None, :]**2
    if r_max is None:
        r_max = np.sqrt(r2.max())

    nbins = int(np.floor((r_max - r_min) / bin_width)) + 1
    if nbins <= 0:
        raise ValueError("Invalid binning: check r_min/r_max/bin_width")

    r = np.sqrt(r2, dtype=np.float64)
    bin_idx = ((r - r_min) / bin_width).astype(np.int64)

    valid = (bin_idx >= 0) & (bin_idx < nbins)

    if threshold is not None:
        valid &= (image >= threshold)

    if mask is not None:
        valid &= mask.astype(bool)

    b = bin_idx[valid].ravel()
    w = image[valid].astype(np.float64, copy=False).ravel()

    if mode == 'sum':
        radial_values = np.bincount(b, weights=w, minlength=nbins).astype(np.float64)
        counts = np.bincount(b, minlength=nbins).astype(np.int64)
    elif mode == 'mean':
        sums = np.bincount(b, weights=w, minlength=nbins).astype(np.float64)
        counts = np.bincount(b, minlength=nbins).astype(np.int64)
        radial_values = sums / np.maximum(counts, 1)
    else:
        raise ValueError("mode must be 'sum' or 'mean'")

    bin_centers = r_min + (np.arange(nbins) + 0.5) * bin_width

    if return_counts:
        return bin_centers, radial_values, counts
    return bin_centers, radial_values

In [ ]:
# def build_crystal_masks(X, Y, crystal_size, inner_gap_lr, inner_gap_tb):
#     half = crystal_size / 2
#     Ri_m_x = half + inner_gap_lr / 2
#     Ri_m_y = half + inner_gap_tb / 2

#     m_top   = (np.abs(X) <= half) & (np.abs(Y - Ri_m_y) <= half)
#     m_bot   = (np.abs(X) <= half) & (np.abs(Y + Ri_m_y) <= half)
#     m_left  = (np.abs(X + Ri_m_x) <= half) & (np.abs(Y) <= half)
#     m_right = (np.abs(X - Ri_m_x) <= half) & (np.abs(Y) <= half)

#     m_lr = m_left | m_right
#     m_tb = m_top | m_bot
#     return m_lr, m_tb, m_left, m_right, m_top, m_bot

def build_crystal_masks(X_m, Y_m, crystal_size, inner_gap_lr, inner_gap_tb):
    """
    X_m, Y_m : arrays in meters (same shape)
    crystal_size : full side length (m) of each square crystal
    inner_gap_lr : gap (m) between left and right crystals (between inner faces)
    inner_gap_tb : gap (m) between top and bottom crystals (between inner faces)

    Returns:
      m_lr, m_tb, m_left, m_right, m_top, m_bot
    """
    half = crystal_size / 2.0
    shift_x = half + inner_gap_lr / 2.0
    shift_y = half + inner_gap_tb / 2.0

    m_top   = (np.abs(X_m) <= half) & (np.abs(Y_m - shift_y) <= half)
    m_bot   = (np.abs(X_m) <= half) & (np.abs(Y_m + shift_y) <= half)
    m_left  = (np.abs(X_m + shift_x) <= half) & (np.abs(Y_m) <= half)
    m_right = (np.abs(X_m - shift_x) <= half) & (np.abs(Y_m) <= half)

    m_lr = m_left | m_right
    m_tb = m_top  | m_bot
    return m_lr, m_tb, m_left, m_right, m_top, m_bot

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# ---- build X,Y in physical units around your chosen center ----
ny, nx = image.shape
center = (2464//2, 2056//2)   # (x0, y0)

x0, y0 = center
x_phys = (np.arange(nx) - x0) * pix_x
y_phys = (np.arange(ny) - y0) * pix_y
X, Y = np.meshgrid(x_phys, y_phys, sparse=False)

# ---- your geometry params ----
crystal_size = 10e-3
inner_gap_lr = 5e-3
inner_gap_tb = 5e-3

m_lr, m_tb, m_left, m_right, m_top, m_bot = build_crystal_masks(
    X, Y, crystal_size, inner_gap_lr, inner_gap_tb
)

# ============================
# IGNORE OVERLAP REGION
# ============================
overlap = m_lr & m_tb
m_lr_use = m_lr & (~overlap)
m_tb_use = m_tb & (~overlap)

# (optional sanity print)
print("LR pixels:", m_lr_use.sum(), "TB pixels:", m_tb_use.sum(), "overlap ignored:", overlap.sum())

# ---- radial binning for LR (excluding overlap) ----
rad_lr, prof_lr = radial_profile_aniso_fast_masked(
    image, center, pix_x, pix_y,
    mask=m_lr_use,
    bin_width=min(pix_x, pix_y),
    mode='sum'
)

# ---- radial binning for TB (excluding overlap) ----
rad_tb, prof_tb = radial_profile_aniso_fast_masked(
    image, center, pix_x, pix_y,
    mask=m_tb_use,
    bin_width=min(pix_x, pix_y),
    mode='sum'
)

# ---- convert r -> time ----
t_lr = time_tilt(rad_lr, ax_angle=10)

delay_amount = 100e-6/c
t_tb = time_tilt(rad_tb, ax_angle=10) - delay_amount  # matches your sim convention (t_crystal = t - delay)

# ---- plot ----
plt.figure()
plt.plot(t_lr, prof_lr, label="Left/Right (no overlap)")
plt.plot(t_tb, prof_tb, label="Top/Bottom (no overlap, shifted)")
plt.xlabel("time (s)")
plt.ylabel("azimuthal sum (counts)")
plt.title("Binned Data: LR vs TB (overlap ignored)")
plt.legend()
plt.show()

In [ ]:
delt12 = 300e-15

def bimodal_gauss(t, loc1, sigma1, A1, loc2, sigma2, A2, offset):
    g1 = A1 * np.exp(-0.5 * ((t - loc1) / sigma1)**2)
    g2 = A2 * np.exp(-0.5 * ((t - loc2) / sigma2)**2)
    return g1 + g2 + offset

# -----------------------
# LEFT/RIGHT REGION
# -----------------------
y = prof_lr
t = t_lr

# OPTIONAL: normalize so the curve shape looks less “noisy” when bins have fewer pixels
# y = y / np.max(y)

p0 = (time_tilt(l_r01*pix_x, ax_angle=10), 30e-6/c, 1,
      time_tilt(l_r01*pix_x, ax_angle=10) + delt12, 30e-6/c, 0.8, 0)

popt, pcov = curve_fit(
    bimodal_gauss, t, y,
    p0=p0,
    maxfev=10_000_000
)

loc1_fit, sigma1_fit, A1_fit, loc2_fit, sigma2_fit, A2_fit, offset_fit = popt

print("\n===== LEFT/RIGHT FIT =====")
print(f"sigma_t (Gaussian) drive beam: {sigma1_fit * 1e15:.2f} fs")
print(f"sigma_t (Gaussian) witness beam: {sigma2_fit * 1e15:.2f} fs")
print(f"drive beam time delay (loc1): {loc1_fit * 1e15:.2f} fs")
print(f"witness beam time delay (loc2): {loc2_fit * 1e15:.2f} fs")
print(f"Δt (loc2-loc1): {(loc2_fit - loc1_fit) * 1e15:.2f} fs")

fit_y = bimodal_gauss(t, *popt)

plt.figure()
plt.plot(t*1e12, y, label="LR binned data")
plt.title("LR Intensity Fit")
plt.ylabel("Intensity (a.u.)")
plt.xlabel("time (ps)")
plt.plot(t*1e12, fit_y, label="LR fit")
plt.legend()
plt.show()

In [ ]:
# -----------------------
# TOP/BOTTOM REGION
# -----------------------
y = prof_tb
t = t_tb   # you already did: time_tilt(rad_tb) - delay_amount

# OPTIONAL: normalize so the curve shape looks less “noisy” when bins have fewer pixels
# y = y / np.max(y)

p0 = (time_tilt(l_r01*pix_x, ax_angle=10), 30e-6/c, 1,
      time_tilt(l_r01*pix_x, ax_angle=10) + delt12, 30e-6/c, 0.8, 0)

popt, pcov = curve_fit(
    bimodal_gauss, t, y,
    p0=p0,
    maxfev=10_000_000
)

loc1_fit, sigma1_fit, A1_fit, loc2_fit, sigma2_fit, A2_fit, offset_fit = popt

print("\n===== TOP/BOTTOM FIT =====")
print(f"sigma_t (Gaussian) drive beam: {sigma1_fit * 1e15:.2f} fs")
print(f"sigma_t (Gaussian) witness beam: {sigma2_fit * 1e15:.2f} fs")
print(f"drive beam time delay (loc1): {loc1_fit * 1e15:.2f} fs")
print(f"witness beam time delay (loc2): {loc2_fit * 1e15:.2f} fs")
print(f"Δt (loc2-loc1): {(loc2_fit - loc1_fit) * 1e15:.2f} fs")

fit_y = bimodal_gauss(t, *popt)

plt.figure()
plt.plot(t*1e12, y, label="TB binned data")
plt.title("TB Intensity Fit")
plt.ylabel("Intensity (a.u.)")
plt.xlabel("time (ps)")
plt.plot(t*1e12, fit_y, label="TB fit")
plt.legend()
plt.show()

In [ ]:
# -----------------------
# Left/Right
# -----------------------
y = prof_lr
t = t_lr   # you already did: time_tilt(rad_tb) - delay_amount

# OPTIONAL: normalize so the curve shape looks less “noisy” when bins have fewer pixels
# y = y / np.max(y)

p0 = (time_tilt(l_r01*pix_x, ax_angle=10), 30e-6/c, 1,
      time_tilt(l_r01*pix_x, ax_angle=10) + delt12, 30e-6/c, 0.8, 0)

popt, pcov = curve_fit(
    bimodal_gauss, t, y,
    p0=p0,
    maxfev=10_000_000
)

loc1_fit, sigma1_fit, A1_fit, loc2_fit, sigma2_fit, A2_fit, offset_fit = popt

print("\n===== LEFT/RIGHT FIT =====")
print(f"sigma_t (Gaussian) drive beam: {sigma1_fit * 1e15:.2f} fs")
print(f"sigma_t (Gaussian) witness beam: {sigma2_fit * 1e15:.2f} fs")
print(f"drive beam time delay (loc1): {loc1_fit * 1e15:.2f} fs")
print(f"witness beam time delay (loc2): {loc2_fit * 1e15:.2f} fs")
print(f"Δt (loc2-loc1): {(loc2_fit - loc1_fit) * 1e15:.2f} fs")

fit_y = bimodal_gauss(t, *popt)

plt.figure()
plt.plot(t*1e12, y, label="LR binned data")
plt.title("LR Intensity Fit")
plt.ylabel("Intensity (a.u.)")
plt.xlabel("time (ps)")
plt.plot(t*1e12, fit_y, label="TB fit")
plt.legend()
plt.show()

### BPM Fitting

In [ ]:
def get_ring_pixels_precise_fast(image, rmin, rmax, x0, y0, pix_x, pix_y, threshold=None):
    """
    rmin/rmax in *pixels* (same as before).
    Anisotropy handled by scaling dy by s = pix_y/pix_x.
    """
    ny, nx = image.shape
    s = float(pix_y) / float(pix_x)  # y pixels are 'bigger' => stretch y in metric

    # bounding box in pixels (safe for anisotropy)
    # In the scaled metric: r^2 = dx^2 + (s*dy)^2 <= rmax^2
    # so |dx| <= rmax, |dy| <= rmax/s
    rmax_y = rmax / max(s, 1e-12)

    x_lo = max(int(np.floor(x0 - rmax)) - 1, 0)
    x_hi = min(int(np.ceil (x0 + rmax)) + 1, nx)
    y_lo = max(int(np.floor(y0 - rmax_y)) - 1, 0)
    y_hi = min(int(np.ceil (y0 + rmax_y)) + 1, ny)

    yy, xx = np.indices((y_hi - y_lo, x_hi - x_lo))
    xx = xx + x_lo
    yy = yy + y_lo

    # corners: (x±0.5, y±0.5) in pixel index units
    dx = np.stack([xx - 0.5 - x0, xx + 0.5 - x0, xx - 0.5 - x0, xx + 0.5 - x0], axis=0)
    dy = np.stack([yy - 0.5 - y0, yy - 0.5 - y0, yy + 0.5 - y0, yy + 0.5 - y0], axis=0)

    # anisotropic radius in "x-pixel metric"
    r2 = dx*dx + (s*dy)*(s*dy)
    rmin2, rmax2 = rmin*rmin, rmax*rmax

    inside = (r2 >= rmin2) & (r2 <= rmax2)
    inside_all = np.all(inside, axis=0)

    if threshold is not None:
        inside_all &= (image[y_lo:y_hi, x_lo:x_hi] > threshold)

    y_pix_roi, x_pix_roi = np.nonzero(inside_all)
    x_pix = x_pix_roi + x_lo
    y_pix = y_pix_roi + y_lo
    intensity = image[y_pix, x_pix]
    return x_pix, y_pix, intensity



def phi1_from_alpha_np(alpha):
    # φ1(alpha) = 1/2 arccos( sin(alpha) / sqrt(1+3 cos^2(alpha)) )
    s = np.sin(alpha)
    c = np.cos(alpha)
    denom = np.sqrt(1.0 + 3.0*c*c)
    arg = np.clip(s / denom, -1.0, 1.0)
    return 0.5 * np.arccos(arg)

@njit(inline='always', fastmath=True)
def _phi1_from_alpha(alpha):
    s = math.sin(alpha)
    c = math.cos(alpha)
    denom = math.sqrt(1.0 + 3.0*c*c)
    arg = s / denom
    if arg > 1.0: arg = 1.0
    elif arg < -1.0: arg = -1.0
    return 0.5 * math.acos(arg)

def intensity_model_3D_piecewise_skew_fast(
    coord, x_b, y_b, A1, A2, alpha1, loc1, amp1, scale,
    region_mask, x0_cent, y0_cent, pix_x, pix_y
):
    x, y = coord
    out = np.empty_like(x, dtype=np.float64)
    eps = 1e-12
    s = float(pix_y) / float(pix_x)

    # --- non-rotated ---
    dx = x - x_b
    dy = (y - y_b) * s     # accounts for uneven pixel size
    b  = dx*dx + dy*dy + eps
    t  = np.sqrt(4.0*dx*dx + dy*dy)

    alpha_ang = np.arctan2(dy, dx)   # <<< uses scaled dy
    phi = phi1_from_alpha_np(alpha_ang)
    out[:] = A1 * (np.sin(phi)**2) * (np.sin(A2 * t / (2.0*b))**2)

    # --- rotated subset accounts for 2 crystals rotated by 90 degrees  ---
    if np.any(region_mask):
        i = region_mask
        dx2 = (y[i] - y_b) * s       # swapped x <- y (scaled)
        dy2 = (x[i] - x_b)           # swapped y <- x (unscaled)
        b2  = dx2*dx2 + dy2*dy2 + eps
        t2  = np.sqrt(4.0*dx2*dx2 + dy2*dy2)

        alpha2 = np.arctan2(dy2, dx2)
        phi2 = phi1_from_alpha_np(alpha2)
        out[i] = A1 * (np.sin(phi2)**2) * (np.sin(A2 * t2 / (2.0*b2))**2)

    # --- radius in same metric (pixels) ---
    r = np.sqrt((x - x0_cent)**2 + ((y - y0_cent)*s)**2)  # also scaled by uneven pixel size
    out *= (amp1 * skewnorm.pdf(r, alpha1, loc1, scale))
    return out


@njit(inline='always')
def _phi(z):
    return math.exp(-0.5*z*z) / math.sqrt(2.0*math.pi)

@njit(inline='always')
def _Phi(z):
    return 0.5*(1.0 + math.erf(z / math.sqrt(2.0)))


@njit(parallel=True, fastmath=True)
def residual_numba_fused(theta, x, y, r, idx_rot, idx_nor, data, s):
    x_b, y_b, A1, A2, alpha_env, loc, amp, sigma = theta
    inv_sigma = 1.0 / sigma
    res = np.empty(x.size, dtype=np.float64)
    eps = 1e-12

    for k in prange(idx_nor.size):
        i = idx_nor[k]
        dx = x[i] - x_b
        dy = (y[i] - y_b) * s   # adjusts for uneven pixel size
        b  = dx*dx + dy*dy + eps
        t  = math.sqrt(4.0*dx*dx + dy*dy)

        alpha_ang = math.atan2(dy, dx)
        phi = _phi1_from_alpha(alpha_ang)
        core = A1 * (math.sin(phi)**2) * (math.sin(A2 * t / (2.0*b))**2)

        z = (r[i] - loc) * inv_sigma
        radial = amp * (2.0 * _phi(z) * _Phi(alpha_env * z)) * inv_sigma
        res[i] = core * radial - data[i]

    for k in prange(idx_rot.size):
        i = idx_rot[k]
        dx = (y[i] - y_b) * s   # adjusts for uneven pixel size (for rotated cystal)
        dy = (x[i] - x_b)       # adjusts for uneven pixel size (for rotated cystal)
        b  = dx*dx + dy*dy + eps
        t  = math.sqrt(4.0*dx*dx + dy*dy)

        alpha_ang = math.atan2(dy, dx)
        phi = _phi1_from_alpha(alpha_ang)
        core = A1 * (math.sin(phi)**2) * (math.sin(A2 * t / (2.0*b))**2)

        z = (r[i] - loc) * inv_sigma
        radial = amp * (2.0 * _phi(z) * _Phi(alpha_env * z)) * inv_sigma
        res[i] = core * radial - data[i]

    return res



def BPM_fit_full_skew_fast_2_aniso(
    image, radius, x0_cent, y0_cent, p0, rmin, rmax, ax, ay,
    pix_x, pix_y, mag=1, threshold=0.001,
    max_pts=120_000, do_plot=True,
    # NEW geometry params (MUST match simulation)
    crystal_size=10e-3,
    inner_gap_lr=5e-3,
    inner_gap_tb=10e-3
):
    ny, nx = image.shape
    pixelsx, pixelsy = nx, ny

    # meters on object plane
    crystx = pixelsx * mag * pix_x / 2.0
    crysty = pixelsy * mag * pix_y / 2.0

    s = float(pix_y) / float(pix_x)
    # axes (no mesh)
    x_axis = np.linspace(-crystx,  crystx,  nx)
    y_axis = np.linspace(-crysty, crysty, ny)
    Xb = x_axis[None, :]
    Yb = y_axis[:, None]

    # region logic
    # -----------------------------
    # NEW region logic (4 crystals)
    # -----------------------------
    # Xb, Yb are already in meters on object plane:
    #   Xb = x_axis[None, :]
    #   Yb = y_axis[:, None]
    #
    # build masks in meters
    m_lr, m_tb, m_left, m_right, m_top, m_bot = build_crystal_masks(
        Xb, Yb, crystal_size=crystal_size,
        inner_gap_lr=inner_gap_lr,
        inner_gap_tb=inner_gap_tb
    )

    # ignore overlap region completely
    overlap = m_lr & m_tb
    m_lr_use = m_lr & (~overlap)
    m_tb_use = m_tb & (~overlap)

    # usable pixels for fitting = ONLY pixels on crystals (excluding overlap)
    usable = m_lr_use | m_tb_use

    # ring pixels
    x_all, y_all, inten_all = get_ring_pixels_precise_fast(
        image, rmin, rmax, x0_cent, y0_cent, pix_x, pix_y, threshold=threshold
    )

    # 2) keep only usable pixels
    m_usable = usable[y_all, x_all]
    x_all, y_all, inten_all = x_all[m_usable], y_all[m_usable], inten_all[m_usable]

    # 3) rotated mask for those pixels (regions 1 & 4)
    rot_mask_full = m_tb_use[y_all, x_all]

    # 4) (optional) subsample BEFORE building idx lists 
    n = x_all.size
    if n > max_pts:
        rng = np.random.default_rng(0)
        idx = rng.choice(n, size=max_pts, replace=False)
        x_all = x_all[idx]; y_all = y_all[idx]; inten_all = inten_all[idx]
        rot_mask_full = rot_mask_full[idx]

    # 5) precompute radii wrt center (float64 for optimizer)
    r_all = np.sqrt((x_all - x0_cent)**2 + ((y_all - y0_cent)*s)**2).astype(np.float64)

    # 6) build indices for numba loops on the SUBSAMPLED mask  
    idx_rot = np.flatnonzero(rot_mask_full.astype(np.uint8)).astype(np.int64)
    idx_nor = np.flatnonzero((~rot_mask_full).astype(np.uint8)).astype(np.int64)

    # 7) ensure contiguous float64
    x_all = np.ascontiguousarray(x_all.astype(np.float64))
    y_all = np.ascontiguousarray(y_all.astype(np.float64))
    inten_all = np.ascontiguousarray(inten_all.astype(np.float64))
    r_all = np.ascontiguousarray(r_all)

    # --- bounds  ---
    offset_margin = radius
    x_min = max(0, x0_cent - offset_margin)
    x_max = min(image.shape[1]-1, x0_cent + offset_margin)
    y_min = max(0, y0_cent - offset_margin)
    y_max = min(image.shape[0]-1, y0_cent + offset_margin)

    bounds = (
        [x_min, y_min,     0.0,  0.0,  -20.0,  rmin,   0.0,   1.0],
        [x_max, y_max, 1e2,   1e2,   20.0,  rmax, 1e3, 5e2]
    )

    # scale intensities once (no double scaling) 
    scale_intensity = float(np.std(inten_all) or 1.0)
    y_scaled = inten_all / scale_intensity

    FWHM_TO_SIG = 1.0 / (2.0*np.sqrt(2.0*np.log(2.0)))

    def residual_wrapper(theta):
        theta = np.asarray(theta, dtype=np.float64).copy()
        theta[7] = max(theta[7]*FWHM_TO_SIG, 1e-9)
        return residual_numba_fused(theta, x_all, y_all, r_all, idx_rot, idx_nor, y_scaled, s)

    # clip p0 and solve
    p0 = np.clip(np.asarray(p0, float), bounds[0], bounds[1])
    # robust scale init
    f_scale = np.median(np.abs(residual_wrapper(p0))) + 1e-12

    res = least_squares(residual_wrapper, p0, bounds=bounds,
                        method="trf", loss="soft_l1",
                        f_scale=f_scale, max_nfev=4000,
                        xtol=1e-6, ftol=1e-6, gtol=1e-6, x_scale='jac')

    params = res.x
    x_b_fit, y_b_fit, A1_fit, A2_fit, alpha1_fit, loc1_fit, amp1_fit, scale_fit = params

    # Optional: visualize sparsely (off by default)
    if do_plot:
        params_plot = params.copy()
        params_plot[7] *= FWHM_TO_SIG
        intensity_fit = intensity_model_3D_piecewise_skew_fast(
            (x_all, y_all), *params_plot, rot_mask_full, x0_cent, y0_cent, pix_x, pix_y
        )
        intensity_fit *= scale_intensity
        extent_mm = [x_axis[0]*1e3, x_axis[-1]*1e3, y_axis[0]*1e3, y_axis[-1]*1e3]
        fit_image = np.zeros_like(image, dtype=np.float64)
        resid_image = np.zeros_like(image, dtype=np.float64)
        x_idx = x_all.astype(np.intp)
        y_idx = y_all.astype(np.intp)
        fit_image[y_idx, x_idx] = intensity_fit
        resid_image[y_idx, x_idx] = (inten_all - intensity_fit)
        fig, axs = plt.subplots(1, 3, figsize=(15, 5))
        axs[0].imshow(image, origin='lower', cmap='inferno'); axs[0].set_title('Original')
        axs[0].scatter([x_b_fit], [y_b_fit], color='cyan', marker='x', label='Fitted Offset')
        
        from matplotlib.patches import Ellipse
        s = pix_y / pix_x
        cx, cy = x0_cent, y0_cent
    

        axs[0].add_patch(Ellipse((cx, cy),
                                width=2*rmin,
                                height=2*(rmin/s),
                                angle=0,
                                ec='w', fc='none', lw=1.2))
        axs[0].add_patch(Ellipse((cx, cy),
                                width=2*rmax,
                                height=2*(rmax/s),
                                angle=0,
                                ec='w', fc='none', lw=1.2))
        axs[0].legend()
        axs[1].imshow(fit_image, origin='lower', cmap='inferno'); axs[1].set_title('Model (sparse)')
        vmax = np.max(np.abs(resid_image))
        im = axs[2].imshow(resid_image, origin='lower', cmap='bwr', vmin=-vmax, vmax=vmax)
        axs[2].set_title('Residual (sparse)'); fig.colorbar(im, ax=axs[2]); plt.tight_layout(); plt.show()

    # meters per pixel
    px = (2.0 * crystx) / nx
    py = (2.0 * crysty) / ny

    x_b_m = (x_b_fit - x0_cent) * px
    y_b_m = (y_b_fit - y0_cent) * py
    diff_x = np.abs(ax) - np.abs(x_b_m)
    diff_y = np.abs(ay) - np.abs(y_b_m)

    print(crystx)
    print(crysty)
    print(f"Fitted Beam Offset: x_b = {x_b_fit:.4f}, y_b = {y_b_fit:.4f}")
    print(f"A_fit = {A1_fit:.3e}, A2_fit = {A2_fit:.3e}")
    print(f"difference between fit and actual x: {diff_x*1e6:.3f} µm")
    print(f"difference between fit and actual y: {diff_y*1e6:.3f} µm")
    print(f"Fit location x_b: {x_b_m*1e6:.3f} µm")
    print(f"Fit location y_b: {y_b_m*1e6:.3f} µm")

    s = pix_y / pix_x
    yy, xx = np.indices(image.shape)
    r_aniso = np.sqrt((xx - x0_cent)**2 + (s*(yy - y0_cent))**2)

    # crude peak estimate near the ring using a mask:
    mask = image > (0.2 * image.max())   # tune threshold
    l_r01_aniso = np.median(r_aniso[mask])  # or np.mean / histogram peak
    print("l_r01_aniso =", l_r01_aniso)


    return x_b_m, y_b_m, ax, ay, diff_x, diff_y

In [ ]:
######################################################################
# Finding the radius of ellipse for mask for fit
######################################################################

s = pix_y / pix_x
yy, xx = np.indices(image.shape)
r = np.sqrt((xx - x0_cent)**2 + (s*(yy - y0_cent))**2)

# only use reasonably bright pixels so background doesn't dominate
m = image > (0.05 * image.max())
r_vals = r[m].ravel()
I_vals = image[m].ravel()

# bin in r and take mean intensity vs r
dr = 1.0  # 1 pixel in the anisotropic metric
rmax_use = np.percentile(r_vals, 99.5)
bins = np.arange(0, rmax_use + dr, dr)

which = np.digitize(r_vals, bins) - 1
valid = (which >= 0) & (which < len(bins)-1)
which = which[valid]
I_vals = I_vals[valid]

sumI = np.bincount(which, weights=I_vals, minlength=len(bins))
cnt  = np.bincount(which, minlength=len(bins))
prof = sumI / np.maximum(cnt, 1)

r_centers = 0.5*(bins[:-1] + bins[1:])

idx_peak = np.argmax(prof)
l_r01_aniso = r_centers[idx_peak]

x0_cent = 2464//2
y0_cent = 2056//2
rmin = l_r01_aniso - 10
rmax = l_r01_aniso + 10

p0 = [
    x0_cent, y0_cent,
    1, 1,
    1,
    l_r01_aniso,  # loc in SAME metric
    1,            # amp
    30,           # sigma
]


x_m, y_m, a_x, a_y, diff_x, diff_y = BPM_fit_full_skew_fast_2_aniso(
    image, l_r01, x0_cent, y0_cent, p0, rmin, rmax, 0, 0,
    pix_x, pix_y, mag=1, threshold=0.001,
    max_pts=120_000, do_plot=True,
    # NEW geometry params (MUST match simulation)
    crystal_size=10e-3,
    inner_gap_lr=5e-3,
    inner_gap_tb=10e-3
)

### BPM Accounting for 2 diffferent radii (still need to fix)

In [ ]:
def get_ring_pixels_precise_fast(image, rmin, rmax, x0, y0, pix_x, pix_y, threshold=None):
    """
    rmin/rmax in *pixels* (same as before).
    Anisotropy handled by scaling dy by s = pix_y/pix_x.
    """
    ny, nx = image.shape
    s = float(pix_y) / float(pix_x)  # y pixels are 'bigger' => stretch y in metric

    # bounding box in pixels (safe for anisotropy)
    # In the scaled metric: r^2 = dx^2 + (s*dy)^2 <= rmax^2
    # so |dx| <= rmax, |dy| <= rmax/s
    rmax_y = rmax / max(s, 1e-12)

    x_lo = max(int(np.floor(x0 - rmax)) - 1, 0)
    x_hi = min(int(np.ceil (x0 + rmax)) + 1, nx)
    y_lo = max(int(np.floor(y0 - rmax_y)) - 1, 0)
    y_hi = min(int(np.ceil (y0 + rmax_y)) + 1, ny)

    yy, xx = np.indices((y_hi - y_lo, x_hi - x_lo))
    xx = xx + x_lo
    yy = yy + y_lo

    # corners: (x±0.5, y±0.5) in pixel index units
    dx = np.stack([xx - 0.5 - x0, xx + 0.5 - x0, xx - 0.5 - x0, xx + 0.5 - x0], axis=0)
    dy = np.stack([yy - 0.5 - y0, yy - 0.5 - y0, yy + 0.5 - y0, yy + 0.5 - y0], axis=0)

    # anisotropic radius in "x-pixel metric"
    r2 = dx*dx + (s*dy)*(s*dy)
    rmin2, rmax2 = rmin*rmin, rmax*rmax

    inside = (r2 >= rmin2) & (r2 <= rmax2)
    inside_all = np.all(inside, axis=0)

    if threshold is not None:
        inside_all &= (image[y_lo:y_hi, x_lo:x_hi] > threshold)

    y_pix_roi, x_pix_roi = np.nonzero(inside_all)
    x_pix = x_pix_roi + x_lo
    y_pix = y_pix_roi + y_lo
    intensity = image[y_pix, x_pix]
    return x_pix, y_pix, intensity



def phi1_from_alpha_np(alpha):
    # φ1(alpha) = 1/2 arccos( sin(alpha) / sqrt(1+3 cos^2(alpha)) )
    s = np.sin(alpha)
    c = np.cos(alpha)
    denom = np.sqrt(1.0 + 3.0*c*c)
    arg = np.clip(s / denom, -1.0, 1.0)
    return 0.5 * np.arccos(arg)

@njit(inline='always', fastmath=True)
def _phi1_from_alpha(alpha):
    s = math.sin(alpha)
    c = math.cos(alpha)
    denom = math.sqrt(1.0 + 3.0*c*c)
    arg = s / denom
    if arg > 1.0: arg = 1.0
    elif arg < -1.0: arg = -1.0
    return 0.5 * math.acos(arg)

def intensity_model_3D_piecewise_skew_fast_locsplit(
    coord, x_b, y_b, A1, A2, alpha_env, loc, amp, sigma_fwhm, dloc_rot,
    region_mask, x0_cent, y0_cent, pix_x, pix_y
):
    x, y = coord
    out = np.empty_like(x, dtype=np.float64)
    eps = 1e-12
    s = float(pix_y) / float(pix_x)

    # core (non-rotated default)
    dx = x - x_b
    dy = (y - y_b) * s
    b  = dx*dx + dy*dy + eps
    t  = np.sqrt(4.0*dx*dx + dy*dy)

    alpha_ang = np.arctan2(dy, dx)
    phi = phi1_from_alpha_np(alpha_ang)
    core = A1 * (np.sin(phi)**2) * (np.sin(A2 * t / (2.0*b))**2)
    out[:] = core

    # overwrite core for rotated subset (TB)
    if np.any(region_mask):
        i = region_mask
        dx2 = (y[i] - y_b) * s
        dy2 = (x[i] - x_b)
        b2  = dx2*dx2 + dy2*dy2 + eps
        t2  = np.sqrt(4.0*dx2*dx2 + dy2*dy2)

        alpha2 = np.arctan2(dy2, dx2)
        phi2 = phi1_from_alpha_np(alpha2)
        out[i] = A1 * (np.sin(phi2)**2) * (np.sin(A2 * t2 / (2.0*b2))**2)

    # radius metric (pixels)
    r = np.sqrt((x - x0_cent)**2 + ((y - y0_cent)*s)**2)

    # convert sigma from FWHM-like param to true sigma (same convention as fitter)
    FWHM_TO_SIG = 1.0 / (2.0*np.sqrt(2.0*np.log(2.0)))
    sigma = max(sigma_fwhm * FWHM_TO_SIG, 1e-9)

    # non-rot envelope
    env = amp * skewnorm.pdf(r, alpha_env, loc, sigma)

    # rotated envelope uses loc + dloc_rot
    if np.any(region_mask):
        i = region_mask
        env[i] = amp * skewnorm.pdf(r[i], alpha_env, loc + dloc_rot, sigma)

    out *= env
    return out


@njit(inline='always')
def _phi(z):
    return math.exp(-0.5*z*z) / math.sqrt(2.0*math.pi)

@njit(inline='always')
def _Phi(z):
    return 0.5*(1.0 + math.erf(z / math.sqrt(2.0)))


@njit(parallel=True, fastmath=True)
def residual_numba_fused_2loc(theta, x, y, r, idx_rot, idx_nor, data, s):
    # theta = [x_b, y_b, A1, A2, alpha_env, loc_lr, loc_tb, amp, sigma]
    x_b, y_b, A1, A2, alpha_env, loc_lr, loc_tb, amp, sigma = theta
    inv_sigma = 1.0 / sigma
    res = np.empty(x.size, dtype=np.float64)
    eps = 1e-12

    # ---- non-rot (LR) uses loc_lr ----
    for k in prange(idx_nor.size):
        i = idx_nor[k]
        dx = x[i] - x_b
        dy = (y[i] - y_b) * s
        b  = dx*dx + dy*dy + eps
        t  = math.sqrt(4.0*dx*dx + dy*dy)

        alpha_ang = math.atan2(dy, dx)
        phi = _phi1_from_alpha(alpha_ang)
        core = A1 * (math.sin(phi)**2) * (math.sin(A2 * t / (2.0*b))**2)

        z = (r[i] - loc_lr) * inv_sigma
        radial = amp * (2.0 * _phi(z) * _Phi(alpha_env * z)) * inv_sigma
        res[i] = core * radial - data[i]

    # ---- rot (TB) uses loc_tb ----
    for k in prange(idx_rot.size):
        i = idx_rot[k]
        dx = (y[i] - y_b) * s
        dy = (x[i] - x_b)
        b  = dx*dx + dy*dy + eps
        t  = math.sqrt(4.0*dx*dx + dy*dy)

        alpha_ang = math.atan2(dy, dx)
        phi = _phi1_from_alpha(alpha_ang)
        core = A1 * (math.sin(phi)**2) * (math.sin(A2 * t / (2.0*b))**2)

        z = (r[i] - loc_tb) * inv_sigma
        radial = amp * (2.0 * _phi(z) * _Phi(alpha_env * z)) * inv_sigma
        res[i] = core * radial - data[i]

    return res


def BPM_fit_full_skew_fast_2_aniso(
    image, radius, x0_cent, y0_cent, p0, rmin, rmax, ax, ay,
    pix_x, pix_y, mag=1, threshold=0.001,
    max_pts=120_000, do_plot=True,
    # NEW geometry params (MUST match simulation)
    crystal_size=10e-3,
    inner_gap_lr=5e-3,
    inner_gap_tb=10e-3
):
    ny, nx = image.shape
    pixelsx, pixelsy = nx, ny

    # meters on object plane
    crystx = pixelsx * mag * pix_x / 2.0
    crysty = pixelsy * mag * pix_y / 2.0

    s = float(pix_y) / float(pix_x)
    # axes (no mesh)
    x_axis = np.linspace(-crystx,  crystx,  nx)
    y_axis = np.linspace(-crysty, crysty, ny)
    Xb = x_axis[None, :]
    Yb = y_axis[:, None]

    # region logic
    # -----------------------------
    # NEW region logic (4 crystals)
    # -----------------------------
    # Xb, Yb are already in meters on object plane:
    #   Xb = x_axis[None, :]
    #   Yb = y_axis[:, None]
    #
    # build masks in meters
    m_lr, m_tb, m_left, m_right, m_top, m_bot = build_crystal_masks(
        Xb, Yb, crystal_size=crystal_size,
        inner_gap_lr=inner_gap_lr,
        inner_gap_tb=inner_gap_tb
    )

    # ignore overlap region completely
    overlap = m_lr & m_tb
    m_lr_use = m_lr & (~overlap)
    m_tb_use = m_tb & (~overlap)

    # usable pixels for fitting = ONLY pixels on crystals (excluding overlap)
    usable = m_lr_use | m_tb_use

    # ring pixels
    x_all, y_all, inten_all = get_ring_pixels_precise_fast(
        image, rmin, rmax, x0_cent, y0_cent, pix_x, pix_y, threshold=threshold
    )

    # 2) keep only usable pixels
    m_usable = usable[y_all, x_all]
    x_all, y_all, inten_all = x_all[m_usable], y_all[m_usable], inten_all[m_usable]

    # 3) rotated mask for those pixels (regions 1 & 4)
    rot_mask_full = m_tb_use[y_all, x_all]

    # 4) (optional) subsample BEFORE building idx lists 
    n = x_all.size
    if n > max_pts:
        rng = np.random.default_rng(0)
        idx = rng.choice(n, size=max_pts, replace=False)
        x_all = x_all[idx]; y_all = y_all[idx]; inten_all = inten_all[idx]
        rot_mask_full = rot_mask_full[idx]

    # 5) precompute radii wrt center (float64 for optimizer)
    r_all = np.sqrt((x_all - x0_cent)**2 + ((y_all - y0_cent)*s)**2).astype(np.float64)

    # 6) build indices for numba loops on the SUBSAMPLED mask  
    idx_rot = np.flatnonzero(rot_mask_full.astype(np.uint8)).astype(np.int64)
    idx_nor = np.flatnonzero((~rot_mask_full).astype(np.uint8)).astype(np.int64)

    # 7) ensure contiguous float64
    x_all = np.ascontiguousarray(x_all.astype(np.float64))
    y_all = np.ascontiguousarray(y_all.astype(np.float64))
    inten_all = np.ascontiguousarray(inten_all.astype(np.float64))
    r_all = np.ascontiguousarray(r_all)

    # --- bounds  ---
    offset_margin = radius
    x_min = max(0, x0_cent - offset_margin)
    x_max = min(image.shape[1]-1, x0_cent + offset_margin)
    y_min = max(0, y0_cent - offset_margin)
    y_max = min(image.shape[0]-1, y0_cent + offset_margin)

    bounds = (
        [x_min, y_min, 0.0, 0.0, -20.0,  rmin,  rmin,  0.0,  1.0],
        [x_max, y_max, 1e2, 1e2,  20.0,  rmax,  rmax,  1e3, 5e2]
    )
    # scale intensities once (no double scaling) 
    scale_intensity = float(np.std(inten_all) or 1.0)
    y_scaled = inten_all / scale_intensity

    FWHM_TO_SIG = 1.0 / (2.0*np.sqrt(2.0*np.log(2.0)))

    def residual_wrapper(theta):
        theta = np.asarray(theta, dtype=np.float64).copy()
        theta[8] = max(theta[8] * FWHM_TO_SIG, 1e-9)   # sigma index shifted to 8
        return residual_numba_fused_2loc(theta, x_all, y_all, r_all,
                                        idx_rot, idx_nor, y_scaled, s)

    # clip p0 and solve
    p0 = list(p0) + [0.0]   # start with same radius in TB and LR
    p0 = np.clip(np.asarray(p0, float), bounds[0], bounds[1])
    # robust scale init
    f_scale = np.median(np.abs(residual_wrapper(p0))) + 1e-12

    res = least_squares(residual_wrapper, p0, bounds=bounds,
                        method="trf", loss="soft_l1",
                        f_scale=f_scale, max_nfev=4000,
                        xtol=1e-6, ftol=1e-6, gtol=1e-6, x_scale='jac')

    params = res.x
    x_b_fit, y_b_fit, A1_fit, A2_fit, alpha_fit, loc_lr_fit, loc_tb_fit, amp_fit, sigma_fit = params
    print("loc_lr =", loc_lr_fit, "loc_tb =", loc_tb_fit, "Δloc =", (loc_tb_fit-loc_lr_fit))

    # Optional: visualize sparsely (off by default)
    if do_plot:
        params_plot = params.copy()
        intensity_fit = intensity_model_3D_piecewise_skew_fast_locsplit(
            (x_all, y_all),
            params_plot[0], params_plot[1], params_plot[2], params_plot[3],
            params_plot[4], params_plot[5], params_plot[6], params_plot[7], params_plot[8],
            rot_mask_full, x0_cent, y0_cent, pix_x, pix_y
        )
        intensity_fit *= scale_intensity
        extent_mm = [x_axis[0]*1e3, x_axis[-1]*1e3, y_axis[0]*1e3, y_axis[-1]*1e3]
        fit_image = np.zeros_like(image, dtype=np.float64)
        resid_image = np.zeros_like(image, dtype=np.float64)
        x_idx = x_all.astype(np.intp)
        y_idx = y_all.astype(np.intp)
        fit_image[y_idx, x_idx] = intensity_fit
        resid_image[y_idx, x_idx] = (inten_all - intensity_fit)
        fig, axs = plt.subplots(1, 3, figsize=(15, 5))
        axs[0].imshow(image, origin='lower', cmap='inferno'); axs[0].set_title('Original')
        axs[0].scatter([x_b_fit], [y_b_fit], color='cyan', marker='x', label='Fitted Offset')
        
        from matplotlib.patches import Ellipse
        s = pix_y / pix_x
        cx, cy = x0_cent, y0_cent
    

        axs[0].add_patch(Ellipse((cx, cy),
                                width=2*rmin,
                                height=2*(rmin/s),
                                angle=0,
                                ec='w', fc='none', lw=1.2))
        axs[0].add_patch(Ellipse((cx, cy),
                                width=2*rmax,
                                height=2*(rmax/s),
                                angle=0,
                                ec='w', fc='none', lw=1.2))
        axs[0].legend()
        axs[1].imshow(fit_image, origin='lower', cmap='inferno'); axs[1].set_title('Model (sparse)')
        vmax = np.max(np.abs(resid_image))
        im = axs[2].imshow(resid_image, origin='lower', cmap='bwr', vmin=-vmax, vmax=vmax)
        axs[2].set_title('Residual (sparse)'); fig.colorbar(im, ax=axs[2]); plt.tight_layout(); plt.show()

    # meters per pixel
    px = (2.0 * crystx) / nx
    py = (2.0 * crysty) / ny

    x_b_m = (x_b_fit - x0_cent) * px
    y_b_m = (y_b_fit - y0_cent) * py
    diff_x = np.abs(ax) - np.abs(x_b_m)
    diff_y = np.abs(ay) - np.abs(y_b_m)

    print(crystx)
    print(crysty)
    print(f"Fitted Beam Offset: x_b = {x_b_fit:.4f}, y_b = {y_b_fit:.4f}")
    print(f"A_fit = {A1_fit:.3e}, A2_fit = {A2_fit:.3e}")
    print(f"difference between fit and actual x: {diff_x*1e6:.3f} µm")
    print(f"difference between fit and actual y: {diff_y*1e6:.3f} µm")
    print(f"Fit location x_b: {x_b_m*1e6:.3f} µm")
    print(f"Fit location y_b: {y_b_m*1e6:.3f} µm")

    s = pix_y / pix_x
    yy, xx = np.indices(image.shape)
    r_aniso = np.sqrt((xx - x0_cent)**2 + (s*(yy - y0_cent))**2)

    # crude peak estimate near the ring using a mask:
    mask = image > (0.2 * image.max())   # tune threshold
    l_r01_aniso = np.median(r_aniso[mask])  # or np.mean / histogram peak
    print("l_r01_aniso =", l_r01_aniso)


    return x_b_m, y_b_m, ax, ay, diff_x, diff_y

In [ ]:
######################################################################
# Finding the radius of ellipse for mask for fit
######################################################################

s = pix_y / pix_x
yy, xx = np.indices(image.shape)
r = np.sqrt((xx - x0_cent)**2 + (s*(yy - y0_cent))**2)

# only use reasonably bright pixels so background doesn't dominate
m = image > (0.05 * image.max())
r_vals = r[m].ravel()
I_vals = image[m].ravel()

# bin in r and take mean intensity vs r
dr = 1.0  # 1 pixel in the anisotropic metric
rmax_use = np.percentile(r_vals, 99.5)
bins = np.arange(0, rmax_use + dr, dr)

which = np.digitize(r_vals, bins) - 1
valid = (which >= 0) & (which < len(bins)-1)
which = which[valid]
I_vals = I_vals[valid]

sumI = np.bincount(which, weights=I_vals, minlength=len(bins))
cnt  = np.bincount(which, minlength=len(bins))
prof = sumI / np.maximum(cnt, 1)

r_centers = 0.5*(bins[:-1] + bins[1:])

idx_peak = np.argmax(prof)
l_r01_aniso = r_centers[idx_peak]

x0_cent = 2464//2
y0_cent = 2056//2
rmin = l_r01_aniso - 10
rmax = l_r01_aniso + 10

p0 = [
    x0_cent, y0_cent,
    1, 1,
    1,
    l_r01_aniso,  # loc in SAME metric
    1,            # amp
    30,           # sigma
]


x_m, y_m, a_x, a_y, diff_x, diff_y = BPM_fit_full_skew_fast_2_aniso(
    image, l_r01, x0_cent, y0_cent, p0, rmin, rmax, 0, 0,
    pix_x, pix_y, mag=1, threshold=0.001,
    max_pts=120_000, do_plot=True,
    # NEW geometry params (MUST match simulation)
    crystal_size=10e-3,
    inner_gap_lr=5e-3,
    inner_gap_tb=5e-3
)

### Scratch

In [ ]:
def build_4crystal_geometry(X, Y,
                            inner_gap,        # total gap (meters)
                            crystal_size=10e-3,
                            delay_topbottom=True,
                            delay_amount=200e-15):

    half = crystal_size / 2

    # Compute center offset from desired gap
    Ri_m = half + inner_gap / 2

    geometry = {}

    # --------------------------
    # TOP crystal
    # --------------------------
    geometry["top"] = {}
    geometry["top"]["mask"] = (
        (np.abs(X) <= half) &
        (np.abs(Y - Ri_m) <= half)
    )
    geometry["top"]["rotated"] = True
    geometry["top"]["delay"] = delay_amount if delay_topbottom else 0

    # --------------------------
    # BOTTOM crystal
    # --------------------------
    geometry["bottom"] = {}
    geometry["bottom"]["mask"] = (
        (np.abs(X) <= half) &
        (np.abs(Y + Ri_m) <= half)
    )
    geometry["bottom"]["rotated"] = True
    geometry["bottom"]["delay"] = delay_amount if delay_topbottom else 0

    # --------------------------
    # LEFT crystal
    # --------------------------
    geometry["left"] = {}
    geometry["left"]["mask"] = (
        (np.abs(X + Ri_m) <= half) &
        (np.abs(Y) <= half)
    )
    geometry["left"]["rotated"] = False
    geometry["left"]["delay"] = 0

    # --------------------------
    # RIGHT crystal
    # --------------------------
    geometry["right"] = {}
    geometry["right"]["mask"] = (
        (np.abs(X - Ri_m) <= half) &
        (np.abs(Y) <= half)
    )
    geometry["right"]["rotated"] = False
    geometry["right"]["delay"] = 0

    return geometry

def plot_crystal_geometry(X, Y, geometry, x, y):

    crystal_map = np.zeros_like(X)

    label_map = {
        "top": 1,
        "bottom": 2,
        "left": 3,
        "right": 4
    }

    for name, crystal in geometry.items():
        crystal_map[crystal["mask"]] = label_map[name]

    plt.figure(figsize=(6,6))
    plt.imshow(crystal_map,
               origin='lower',
               extent=[x.min()*1e3, x.max()*1e3,
                       y.min()*1e3, y.max()*1e3],
               aspect='equal')

    plt.colorbar(label="1=Top, 2=Bottom, 3=Left, 4=Right")
    plt.xlabel("x (mm)")
    plt.ylabel("y (mm)")
    plt.title("Crystal Geometry")
    plt.show()

def plot_crystal_geometry(X, Y, geometry, x, y):

    crystal_map = np.zeros_like(X)

    label_map = {
        "top": 1,
        "bottom": 2,
        "left": 3,
        "right": 4
    }

    for name, crystal in geometry.items():
        crystal_map[crystal["mask"]] = label_map[name]

    plt.figure(figsize=(6,6))
    plt.imshow(crystal_map,
               origin='lower',
               extent=[x.min()*1e3, x.max()*1e3,
                       y.min()*1e3, y.max()*1e3],
               aspect='equal')

    plt.colorbar(label="1=Top, 2=Bottom, 3=Left, 4=Right")
    plt.xlabel("x (mm)")
    plt.ylabel("y (mm)")
    plt.title("Crystal Geometry")
    plt.show()

# Simulation size parameters
pixelsx = 2464
pixelsy = 2056
Ro = 13.2  # mm

crystx = Ro * 1e-3
crysty = Ro * 1e-3

x = np.linspace(-crystx, crystx, pixelsx)
y = np.linspace(-crysty, crysty, pixelsy)
X, Y = np.meshgrid(x, y)



In [ ]:
geometry = build_4crystal_geometry(
    X, Y,
    inner_gap=5e-3,       # 2 mm
    crystal_size=10e-3
)
plot_crystal_geometry(X, Y, geometry, x, y)

In [ ]:
######################################################################################################
# Simulation with pulse front tilt calculation
######################################################################################################

def EOS_sim_2b_finite(r01x, r01y,r02x,r02y, time_delay,
               Q1 = 1000e-12,Q2 = 800e-12,
               sig_1 = 15e-6/3e8,sig_2 = 15e-6/3e8, delt12 = 400e-15,
               dcry = 100e-6, theta = 5, 
               pixelsx = 2856, pixelsy = 2856,
               Ri = 5.2, Ro = 13.2, laser = 800e-9,sig_eff_laser = 50e-15,inner_gap = 2e-3,crystal_size = 10e-3,
               delay_topbottom = True,delay_amount = 200e-15):
    """  

    This function produces the laser intensity profile on a camera from EO crystal response from ebeam w/ offset
    - r01x,r01y: drive beam offset in m 
    - r02x,r02y: witness beam offset in m 
    - time_delay: intitial time delay in s
    - pixel_sizex/y: pixel size on camera in x/y (m)
    - magn: magnification from crystal to camera
    - Q1, Q2: ebeam charge in C
    - sig_1,sig_2: sig of each bunch in s
    - delt12: time delay between drive and witness beam in s
    - dcry: crystal thickness in m
    - theta: laser ange (deg)
    - pixelsx,pixelsy: number if pixels in x and y (take num of camera pixels)
    - Ri,Ro: inner and outer radius of laser donut in mm 
    - laser: laser wavelength in m
    - cryst_geo: True is for updated crystal orientation, False is for 1 orientation
    
    
    """
    ########################################################################################
    # Creating mesh grid and defining laser and ebeam location "on the crystal"
    ########################################################################################
    #constants
    c = 2.99792458e8 #m/s
    hbar = 1.055e-34 #J s
    e0 = 8.85e-12 #F/m

    # Defines the meshgrid with dimensions on the crystal
    crystx = Ro*1e-3
    crysty = Ro*1e-3

    x = np.linspace(-crystx, crystx, pixelsx)
    y = np.linspace(-crysty, crysty, pixelsy)
    print(f"pixel size x: {x[1]-x[0]}")
    print(f"pixel size y: {y[1]-y[0]}")

    X, Y = np.meshgrid(x, y, sparse=False)
    Rl = np.sqrt(X**2 + Y**2)

    # ---------------------------------------------------
# Build 4-crystal cross geometry (aligned to grid)
# ---------------------------------------------------

    half = crystal_size / 2
    Ri_m = half + inner_gap / 2  # center offset from explicit gap

    geometry = {}

    # TOP
    geometry["top"] = {
        "mask": (np.abs(X) <= half) &
                (np.abs(Y - Ri_m) <= half),
        "rotated": True,
        "delay": delay_amount if delay_topbottom else 0
    }

    # BOTTOM
    geometry["bottom"] = {
        "mask": (np.abs(X) <= half) &
                (np.abs(Y + Ri_m) <= half),
        "rotated": True,
        "delay": delay_amount if delay_topbottom else 0
    }

    # LEFT
    geometry["left"] = {
        "mask": (np.abs(X + Ri_m) <= half) &
                (np.abs(Y) <= half),
        "rotated": False,
        "delay": 0
    }

    # RIGHT
    geometry["right"] = {
        "mask": (np.abs(X - Ri_m) <= half) &
                (np.abs(Y) <= half),
        "rotated": False,
        "delay": 0
    }

    print(f"defined meshgrid")
    plot_crystal_geometry(X, Y, geometry, x, y)
    # Defines the relations between time and donut radius (pulse front tilt)
    Ric = Ri*1e-3
    theta = np.radians(theta)
    t2 = (Rl-Ric) * np.sin(theta)/(c* np.cos(theta)**2)
    ax_angle = 10
    θ1 = np.radians(ax_angle) # axicon angle 
    ng =  1.4671
    n = 1.4533
    θ3 = (θ1*(n-1)) # convergence angle
    θ5 = np.pi/2 - (θ1*(n-1) + np.arctan((1- (θ1**2 * ng*(n-1)))/(θ1*(ng-1))))
    
    Rlaser =  (Rl-Ric)
    # Rlaser = np.maximum(Rl - Ric, 0)
    t = Rlaser * (θ5 + θ1*(n-1))/c
    

    # t = time_tilt((Rl - Ric))
    print(f"time min {np.min(t[2856//2])*1e12} time max {np.max(t[2856//2])*1e12}")
    print(f"time min {np.min(t2[2856//2])*1e12} time max {np.max(t2[2856//2])*1e12}")    
    print(f"{Rl[2856//2][0]} {Rl[2856//2][-1]}") 

    # angle alpha with respect to ebeam location (Casabouloni 2008)
    alphab1 = np.arctan2(Y-r01y,X-r01x) # dependent on drive beam location
    alphab2 = np.arctan2(Y-r02y,X-r02x) # dependent on witness beam location

    alphab1 = np.mod(alphab1, 2 * np.pi) # angle (0,2pi)
    alphab2 = np.mod(alphab2, 2 * np.pi) # angle (0,2pi)

    # radius mapped to location on crystal taking offset into account
    Rlo1 = np.sqrt((X-r01x)**2+(Y-r01y)**2) # dependent on drive beam location
    Rlo2 = np.sqrt((X-r02x)**2+(Y-r02y)**2) # dependent on witness beam location

    ratio1 = Rlo1 / Rl
    ratio2 = Rlo2 / Rl

   # angle alpha with respect to ebeam location (Casabouloni 2008)
    alphab1 = np.arctan2(Y-r01y,X-r01x) # dependent on drive beam location
    alphab2 = np.arctan2(Y-r02y,X-r02x) # dependent on witness beam location

    alphab1 = np.mod(alphab1, 2 * np.pi) # angle (0,2pi)
    alphab2 = np.mod(alphab2, 2 * np.pi) # angle (0,2pi)

    # radius mapped to location on crystal taking offset into account
    Rlo1 = np.sqrt((X-r01x)**2+(Y-r01y)**2) # dependent on drive beam location
    Rlo2 = np.sqrt((X-r02x)**2+(Y-r02y)**2) # dependent on witness beam location

    eps = 1e-12
    ratio1 = Rlo1 / (Rl + eps)
    ratio2 = Rlo2 / (Rl + eps)


    # Prepare arrays
    Gamma = np.zeros_like(alphab1)
    alpha_rot1 = (alphab1 + np.pi/2) % (2*np.pi)
    alpha_rot2 = (alphab2 + np.pi/2) % (2*np.pi)

    # Store per-crystal fields
    total_field1_dict = {}
    total_field2_dict = {}

    # -------------------------------------------------
    # Step 1: compute per-crystal E-field including delay
    # -------------------------------------------------
    for name, crystal in geometry.items():
        mask = crystal["mask"]
        
        # Apply crystal-specific delay
        if crystal["delay"] != 0:
            t_crystal = t - crystal["delay"]
        else:
            t_crystal = t

        # Compute 1D E-fields for drive/witness
        Er1_crystal = Er1t(Rl, t_crystal, time_delay, sig_1, Q1)
        Er2_crystal = Er2t(Rl, t_crystal, time_delay, delt12, sig_2, Q2)
        Er1_crystal = np.nan_to_num(Er1_crystal, nan=0.0, posinf=0.0, neginf=0.0)
        Er2_crystal = np.nan_to_num(Er2_crystal, nan=0.0, posinf=0.0, neginf=0.0)

        # FFT -> frequency domain
        f1, Erf1, dt = FThz1d(t_crystal, Er1_crystal)
        f2, Erf2, dt = FThz1d(t_crystal, Er2_crystal)

        # Apply FAT/geometric response
        Gd1 = Gd_interp(f1, G(f1, dcry, vg_opt_eff(laser, theta), sig_eff_laser))
        Gd2 = Gd_interp(f2, G(f2, dcry, vg_opt_eff(laser, theta), sig_eff_laser))

        GEOd1 = GEOd(f1, Gd1)
        GEOd2 = GEOd(f2, Gd2)
        GEOdref1 = GEO_ref(f1, dcry)
        GEOdref2 = GEO_ref(f2, dcry)

        FEeff1 = Erf1 * GEOd1
        FEeff2 = Erf2 * GEOd2
        FEreff1 = GEOdref1 * FEeff1
        FEreff2 = GEOdref2 * FEeff2

        # IFFT -> back to time domain
        trans_field1 = ifft_response(f1, FEeff1, dt)
        trans_field2 = ifft_response(f2, FEeff2, dt)
        ref_field1   = ifft_response(f1, FEreff1, dt)
        ref_field2   = ifft_response(f2, FEreff2, dt)

        # Recast 1D -> 2D, scale by Rl
        total_field1_dict[name] = recast_1d_to_2d(Rl, trans_field1 + ref_field1) * 1/ratio1
        total_field2_dict[name] = recast_1d_to_2d(Rl, trans_field2 + ref_field2) * 1/ratio2
        
        total_field1_dict[name][t_crystal < 0] = 0
        total_field2_dict[name][t_crystal < 0] = 0
    # -------------------------------------------------
    # Step 2: compute Gamma per crystal with rotation
    # -------------------------------------------------
    for name, crystal in geometry.items():
        mask = crystal["mask"]
        field1 = total_field1_dict[name]
        field2 = total_field2_dict[name]

        if crystal["rotated"]:
            a1 = alpha_rot1
            a2 = alpha_rot2
        else:
            a1 = alphab1
            a2 = alphab2

        # Only inside the crystal mask
        Gamma[mask] += Gamma_1(a1[mask], laser, dcry, field1[mask])
        Gamma[mask] += Gamma_1(a2[mask], laser, dcry, field2[mask])

    Intensity = np.sin(Gamma/2)**2
        

    fig, axes = plt.subplots(1, 2, figsize=(15, 5), constrained_layout=True)
    im0 = axes[0].imshow(Intensity*100,
                        cmap='inferno',
                        origin='lower',
                        extent=[x.min()*1e3, x.max()*1e3, y.min()*1e3, y.max()*1e3],
                        aspect='equal') #*image
    axes[0].set_title('Intensity')
    axes[0].set_xlabel('x (mm)')
    axes[0].set_ylabel('y (mm)')
    plt.colorbar(im0, ax=axes[0], fraction=0.046, pad=0.04, label='% Transmission at Detector')

    # --------------------------------------------------------------------
    # Right: Er field
    im1 = axes[1].imshow(Gamma*1e3,
                        cmap='inferno',
                        origin='lower',
                        extent=[x.min()*1e3, x.max()*1e3, y.min()*1e3, y.max()*1e3],
                        aspect='equal')
    axes[1].set_title(r'Phase Shift (Γ)')
    axes[1].set_xlabel('x (mm)')
    axes[1].set_ylabel('y (mm)')
    plt.colorbar(im1, ax=axes[1],label='mrad',fraction=0.046, pad=0.04)
    plt.show()
    print(f"Maximum Intensity {np.max(Intensity)*100} %")

    del X,Y, x,y,alphab1,alphab2 #,alpha_final1
    gc.collect()

    return Intensity, Gamma , t#*image


In [ ]:
r01x = 0e-3 #drive
r01y =0e-3 #drive
r02x = 0e-3 #witness
r02y = 0e-3 #witness
time_delay =5e-13 
sig_z2_thz = 40e-6 # length (m)
sig_t2_thz = sig_z2_thz/c 

Intensity, Gamma,t =  EOS_sim_2b_finite(r01x, r01y,r02x,r02y, time_delay,
               Q1 = 1000e-12,Q2 = 800e-12,
               sig_1 = 15e-6/3e8,sig_2 = 15e-6/3e8, delt12 = 400e-15,
               dcry = 100e-6, theta = 5, 
               pixelsx = 2645, pixelsy = 2057,
               Ri = 3, Ro = 13.2, laser = 800e-9,sig_eff_laser = 50e-15,inner_gap = 2e-3,crystal_size = 10e-3,
               delay_topbottom = True,delay_amount = 200e-15)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap

def plot_overlap_geometry(x, y, m_top, m_bot, m_left, m_right, title="Crystal Geometry"):
    """
    Visualize 4 rectangle masks.
    0=none, 1=top, 2=bottom, 3=left, 4=right.
    If overlaps exist, later assignments overwrite earlier ones (just for display).
    """
    geo = np.zeros(m_top.shape, dtype=np.int8)

    geo[m_top] = 1
    geo[m_bot] = 2
    geo[m_left] = 3
    geo[m_right] = 4

    cmap = ListedColormap(["black", "tab:blue", "tab:green", "tab:orange", "tab:red"])

    plt.figure(figsize=(6, 5), constrained_layout=True)
    plt.imshow(
        geo,
        origin="lower",
        extent=[x.min()*1e3, x.max()*1e3, y.min()*1e3, y.max()*1e3],
        aspect="equal",
        cmap=cmap,
        vmin=0, vmax=4
    )
    plt.title(title)
    plt.xlabel("x (mm)")
    plt.ylabel("y (mm)")
    cbar = plt.colorbar(ticks=[0, 1, 2, 3, 4])
    cbar.ax.set_yticklabels(["none", "top", "bottom", "left", "right"])
    plt.show()

In [ ]:
def EOS_sim_2b_finite(r01x, r01y, r02x, r02y, time_delay,
                      pix_sizex=3.45e-6, pix_sizey=3.45e-6, magn=2,
                      Q1=1000e-12, Q2=800e-12,
                      sig_1=15e-6/3e8, sig_2=15e-6/3e8, delt12=400e-15,
                      dcry=100e-6, theta=5,
                      pixelsx=2856, pixelsy=2856,
                      Ri=5.2, Ro=13.2, laser=800e-9, sig_eff_laser=50e-15,
                      cryst_geo=True,
                      # --- NEW geometry params ---
                      crystal_size=10e-3, inner_gap=2e-3,
                      delay_topbottom=True, delay_amount=200e-15,
                      # keep your old edge masking switch if you still want it
                      edges=False):
    """
    Same calculation as old version, but:
      - use 4 overlapping crystals (rectangles)
      - top/bottom rotated by 90 deg
      - top/bottom get an additional time delay (delay_amount)
      - left/right do NOT
    """

    # ----------------------------
    # meshgrid on crystal
    # ----------------------------
    crystx = Ro * 1e-3
    crysty = Ro * 1e-3

    x = np.linspace(-crystx, crystx, pixelsx)
    y = np.linspace(-crysty, crysty, pixelsy)
    X, Y = np.meshgrid(x, y, sparse=False)
    Rl = np.sqrt(X**2 + Y**2)

    # ----------------------------
    # pulse-front tilt time map (unchanged)
    # ----------------------------
    Ric = Ri * 1e-3
    theta_rad = np.radians(theta)

    ax_angle = 10
    θ1 = np.radians(ax_angle)
    ng = 1.4671
    n_ = 1.4533
    θ5 = np.pi/2 - (θ1*(n_-1) + np.arctan((1 - (θ1**2 * ng*(n_-1))) / (θ1*(ng-1))))

    Rlaser = (Rl - Ric)
    t = Rlaser * (θ5 + θ1*(n_-1)) / c

    # ----------------------------
    # angles / ratios (unchanged)
    # ----------------------------
    alphab1 = np.mod(np.arctan2(Y - r01y, X - r01x), 2*np.pi)
    alphab2 = np.mod(np.arctan2(Y - r02y, X - r02x), 2*np.pi)

    Rlo1 = np.sqrt((X - r01x)**2 + (Y - r01y)**2)
    Rlo2 = np.sqrt((X - r02x)**2 + (Y - r02y)**2)

    ratio1 = Rlo1 / Rl
    ratio2 = Rlo2 / Rl

    ratio1 = np.nan_to_num(ratio1, nan=0.0, posinf=0.0, neginf=0.0)
    ratio2 = np.nan_to_num(ratio2, nan=0.0, posinf=0.0, neginf=0.0)

    # rotated alpha (90 deg)
    alpha1_rot = (alphab1 + np.pi/2) % (2*np.pi)
    alpha2_rot = (alphab2 + np.pi/2) % (2*np.pi)

    # ---------------------------------------------------------
    # helper: compute total_field1,total_field2 for a time_delay
    # using EXACT SAME pipeline you had before
    # ---------------------------------------------------------
    def compute_total_fields(time_delay_eff):
        # FFT for each bunch separately (same as old code)
        f1, Erf1, dt = FThz1d(t, Er1t(Rl, t, time_delay_eff, sig_1, Q1))
        f2, Erf2, dt = FThz1d(t, Er2t(Rl, t, time_delay_eff, delt12, sig_2, Q2))

        Gd1 = Gd_interp(f1, G(f1, dcry, vg_opt_eff(laser, theta_rad), sig_eff_laser))
        Gd2 = Gd_interp(f2, G(f2, dcry, vg_opt_eff(laser, theta_rad), sig_eff_laser))

        GEOd1 = GEOd(f1, Gd1)
        GEOd2 = GEOd(f2, Gd2)

        GEOdref1 = GEO_ref(f1, dcry)
        GEOdref2 = GEO_ref(f2, dcry)

        FEeff1 = Erf1 * GEOd1
        FEeff2 = Erf2 * GEOd2

        FEreff1 = GEOdref1 * FEeff1
        FEreff2 = GEOdref2 * FEeff2

        trans_field1 = ifft_response(f1, FEeff1, dt)
        trans_field2 = ifft_response(f2, FEeff2, dt)

        ref_field1 = ifft_response(f1, FEreff1, dt)
        ref_field2 = ifft_response(f2, FEreff2, dt)

        trans_field1f = recast_1d_to_2d(Rl, trans_field1) * (1/ratio1)
        trans_field2f = recast_1d_to_2d(Rl, trans_field2) * (1/ratio2)

        ref_fieldf1 = recast_1d_to_2d(Rl, ref_field1) * (1/ratio1)
        ref_fieldf2 = recast_1d_to_2d(Rl, ref_field2) * (1/ratio2)

        total_field1 = trans_field1f + ref_fieldf1
        total_field2 = trans_field2f + ref_fieldf2

        total_field1[t < 0] = 0
        total_field2[t < 0] = 0

        total_field1 = np.nan_to_num(total_field1, nan=0.0, posinf=0.0, neginf=0.0)
        total_field2 = np.nan_to_num(total_field2, nan=0.0, posinf=0.0, neginf=0.0)
        return total_field1, total_field2

    # nominal fields (left/right)
    total_field1_nom, total_field2_nom = compute_total_fields(time_delay)

    # delayed fields (top/bottom)
    if delay_topbottom and delay_amount != 0:
        total_field1_del, total_field2_del = compute_total_fields(time_delay + delay_amount)
    else:
        total_field1_del, total_field2_del = total_field1_nom, total_field2_nom

    # ----------------------------
    # build 4-rectangle geometry
    # ----------------------------
    half = crystal_size / 2
    Ri_m = half + inner_gap / 2  # center offset

    m_top = (np.abs(X) <= half) & (np.abs(Y - Ri_m) <= half)
    m_bot = (np.abs(X) <= half) & (np.abs(Y + Ri_m) <= half)
    m_left = (np.abs(X + Ri_m) <= half) & (np.abs(Y) <= half)
    m_right = (np.abs(X - Ri_m) <= half) & (np.abs(Y) <= half)

    # optional: restrict to union of crystals (outside = 0)
    m_union = m_top | m_bot | m_left | m_right

    plot_overlap_geometry(x, y, m_top, m_bot, m_left, m_right,
                      title="Crystal Geometry (top/bottom rotated, top/bottom delayed)")
    # ----------------------------
    # compose Gamma exactly like old style (region-based)
    # ----------------------------
    # Gamma_map = np.zeros_like(Rl)

    # # left/right: no rotation, nominal fields
    # Gamma_map[m_left]  = Gamma_1(alphab1[m_left],  laser, dcry, total_field1_nom[m_left]) + \
    #                      Gamma_1(alphab2[m_left],  laser, dcry, total_field2_nom[m_left])
    # Gamma_map[m_right] = Gamma_1(alphab1[m_right], laser, dcry, total_field1_nom[m_right]) + \
    #                      Gamma_1(alphab2[m_right], laser, dcry, total_field2_nom[m_right])

    # # top/bottom: rotated alpha, delayed fields
    # Gamma_map[m_top] = Gamma_1(alpha1_rot[m_top], laser, dcry, total_field1_del[m_top]) + \
    #                    Gamma_1(alpha2_rot[m_top], laser, dcry, total_field2_del[m_top])
    # Gamma_map[m_bot] = Gamma_1(alpha1_rot[m_bot], laser, dcry, total_field1_del[m_bot]) + \
    #                    Gamma_1(alpha2_rot[m_bot], laser, dcry, total_field2_del[m_bot])

    # # outside crystals -> 0
    # Gamma_map[~m_union] = 0.0

    Gamma_map = np.zeros_like(Rl)
    Intensity = np.zeros_like(Rl)

    # ---- LEFT (φ1 + Gamma_1), nominal fields ----
    g1 = Gamma_1(alphab1[m_left], laser, dcry, total_field1_nom[m_left])
    g2 = Gamma_1(alphab2[m_left], laser, dcry, total_field2_nom[m_left])
    Gamma_map[m_left] = g1 + g2
    Intensity[m_left] = (np.sin(φ1(alphab1[m_left]))**2) * (np.sin(g1/2)**2) + \
                        (np.sin(φ1(alphab2[m_left]))**2) * (np.sin(g2/2)**2)

    # ---- RIGHT (φ1 + Gamma_1), nominal fields ----
    g1 = Gamma_1(alphab1[m_right], laser, dcry, total_field1_nom[m_right])
    g2 = Gamma_1(alphab2[m_right], laser, dcry, total_field2_nom[m_right])
    Gamma_map[m_right] = g1 + g2
    Intensity[m_right] = (np.sin(φ1(alphab1[m_right]))**2) * (np.sin(g1/2)**2) + \
                        (np.sin(φ1(alphab2[m_right]))**2) * (np.sin(g2/2)**2)

    # ---- TOP (φ2 + Gamma_2), delayed fields ----
    g1 = Gamma_2(alphab1[m_top], laser, dcry, total_field1_del[m_top])
    g2 = Gamma_2(alphab2[m_top], laser, dcry, total_field2_del[m_top])
    Gamma_map[m_top] = g1 + g2
    Intensity[m_top] = (np.sin(φ2(alphab1[m_top]))**2) * (np.sin(g1/2)**2) + \
                    (np.sin(φ2(alphab2[m_top]))**2) * (np.sin(g2/2)**2)

    # ---- BOTTOM (φ2 + Gamma_2), delayed fields ----
    g1 = Gamma_2(alphab1[m_bot], laser, dcry, total_field1_del[m_bot])
    g2 = Gamma_2(alphab2[m_bot], laser, dcry, total_field2_del[m_bot])
    Gamma_map[m_bot] = g1 + g2
    Intensity[m_bot] = (np.sin(φ2(alphab1[m_bot]))**2) * (np.sin(g1/2)**2) + \
                    (np.sin(φ2(alphab2[m_bot]))**2) * (np.sin(g2/2)**2)

    # outside crystals -> 0
    Gamma_map[~m_union] = 0.0
    Intensity[~m_union] = 0.0

    Intensity = np.sin(Gamma_map/2)**2

    # plot
    fig, axes = plt.subplots(1, 2, figsize=(15, 5), constrained_layout=True)

    im0 = axes[0].imshow(Intensity*100, cmap='inferno', origin='lower',
                         extent=[x.min()*1e3, x.max()*1e3, y.min()*1e3, y.max()*1e3],
                         aspect='equal')
    axes[0].set_title('Intensity')
    axes[0].set_xlabel('x (mm)')
    axes[0].set_ylabel('y (mm)')
    plt.colorbar(im0, ax=axes[0], fraction=0.046, pad=0.04, label='% Transmission at Detector')

    im1 = axes[1].imshow(Gamma_map*1e3, cmap='inferno', origin='lower',
                         extent=[x.min()*1e3, x.max()*1e3, y.min()*1e3, y.max()*1e3],
                         aspect='equal')
    axes[1].set_title(r'Phase Shift (Γ)')
    axes[1].set_xlabel('x (mm)')
    axes[1].set_ylabel('y (mm)')
    plt.colorbar(im1, ax=axes[1], label='mrad', fraction=0.046, pad=0.04)

    plt.show()
    print(f"Maximum Intensity {np.max(Intensity)*100:.3f} %")

    del X, Y, x, y, alphab1, alphab2
    gc.collect()

    return Intensity, Gamma_map, t

In [ ]:
EOS_sim_2b_finite(r01x, r01y, r02x, r02y, time_delay,
                      pix_sizex=3.45e-6, pix_sizey=3.45e-6, magn=2,
                      Q1=1.6e-9, Q2=0e-12,
                      sig_1=15e-6/3e8, sig_2=15e-6/3e8, delt12=400e-15,
                      dcry=100e-6, theta=5,
                      pixelsx=2856, pixelsy=2856,
                      Ri=3, Ro=13.2, laser=800e-9, sig_eff_laser=50e-15,
                      cryst_geo=True,
                      # --- NEW geometry params ---
                      crystal_size=10e-3, inner_gap=2e-3,
                      delay_topbottom=True, delay_amount=200e-15,
                      # keep your old edge masking switch if you still want it
                      edges=False)

In [ ]:
def EOS_sim_2b_finite(r01x, r01y, r02x, r02y, time_delay,
                      pix_sizex=3.45e-6, pix_sizey=3.45e-6, magn=2,
                      Q1=1000e-12, Q2=800e-12,
                      sig_1=15e-6/3e8, sig_2=15e-6/3e8, delt12=400e-15,
                      dcry=100e-6, theta=5,
                      pixelsx=2856, pixelsy=2856,
                      Ri=5.2, Ro=13.2, laser=800e-9, sig_eff_laser=50e-15,
                      cryst_geo=True,
                      # --- geometry params ---
                      crystal_size=10e-3,
                      inner_gap_lr=2e-3,   # gap between LEFT and RIGHT crystals
                      inner_gap_tb=2e-3,   # gap between TOP and BOTTOM crystals
                      # --- delay params ---
                      delay_topbottom=True, delay_amount=200e-15,
                      # optional edge masking
                      edges=False):
    """
    Same calculation as old version, but with:
      - 4 rectangular "crystals"
      - left/right: NOT rotated, NOT delayed
      - top/bottom: rotated (use φ2 + Gamma_2), additionally delayed
      - overlaps: contributions ADD (so overlap regions contain both signals)
      - independent gaps: inner_gap_lr and inner_gap_tb
    """

    # ----------------------------
    # meshgrid on crystal
    # ----------------------------
    crystx = Ro * 1e-3
    crysty = Ro * 1e-3

    x = np.linspace(-crystx, crystx, pixelsx)
    y = np.linspace(-crysty, crysty, pixelsy)
    X, Y = np.meshgrid(x, y, sparse=False)
    Rl = np.sqrt(X**2 + Y**2)

    # ----------------------------
    # pulse-front tilt time map (unchanged)
    # ----------------------------
    Ric = Ri * 1e-3
    theta_rad = np.radians(theta)

    ax_angle = 10
    θ1 = np.radians(ax_angle)
    ng = 1.4671
    n_ = 1.4533
    θ5 = np.pi/2 - (θ1*(n_-1) + np.arctan((1 - (θ1**2 * ng*(n_-1))) / (θ1*(ng-1))))

    Rlaser = (Rl - Ric)
    t = Rlaser * (θ5 + θ1*(n_-1)) / c

    # ----------------------------
    # angles / ratios (unchanged)
    # ----------------------------
    alphab1 = np.mod(np.arctan2(Y - r01y, X - r01x), 2*np.pi)
    alphab2 = np.mod(np.arctan2(Y - r02y, X - r02x), 2*np.pi)

    Rlo1 = np.sqrt((X - r01x)**2 + (Y - r01y)**2)
    Rlo2 = np.sqrt((X - r02x)**2 + (Y - r02y)**2)

    ratio1 = Rlo1 / Rl
    ratio2 = Rlo2 / Rl
    ratio1 = np.nan_to_num(ratio1, nan=0.0, posinf=0.0, neginf=0.0)
    ratio2 = np.nan_to_num(ratio2, nan=0.0, posinf=0.0, neginf=0.0)

    # ---------------------------------------------------------
    # helper: compute total_field1,total_field2 for a time_delay
    # using EXACT SAME pipeline you had before
    # ---------------------------------------------------------
    def compute_total_fields(time_delay_eff):
        f1, Erf1, dt = FThz1d(t, Er1t(Rl, t, time_delay_eff, sig_1, Q1))
        f2, Erf2, dt = FThz1d(t, Er2t(Rl, t, time_delay_eff, delt12, sig_2, Q2))

        Gd1 = Gd_interp(f1, G(f1, dcry, vg_opt_eff(laser, theta_rad), sig_eff_laser))
        Gd2 = Gd_interp(f2, G(f2, dcry, vg_opt_eff(laser, theta_rad), sig_eff_laser))

        GEOd1 = GEOd(f1, Gd1)
        GEOd2 = GEOd(f2, Gd2)

        GEOdref1 = GEO_ref(f1, dcry)
        GEOdref2 = GEO_ref(f2, dcry)

        FEeff1 = Erf1 * GEOd1
        FEeff2 = Erf2 * GEOd2

        FEreff1 = GEOdref1 * FEeff1
        FEreff2 = GEOdref2 * FEeff2

        trans_field1 = ifft_response(f1, FEeff1, dt)
        trans_field2 = ifft_response(f2, FEeff2, dt)

        ref_field1 = ifft_response(f1, FEreff1, dt)
        ref_field2 = ifft_response(f2, FEreff2, dt)

        trans_field1f = recast_1d_to_2d(Rl, trans_field1) * (1/ratio1)
        trans_field2f = recast_1d_to_2d(Rl, trans_field2) * (1/ratio2)

        ref_fieldf1 = recast_1d_to_2d(Rl, ref_field1) * (1/ratio1)
        ref_fieldf2 = recast_1d_to_2d(Rl, ref_field2) * (1/ratio2)

        total_field1 = trans_field1f + ref_fieldf1
        total_field2 = trans_field2f + ref_fieldf2

        total_field1[t < 0] = 0
        total_field2[t < 0] = 0

        total_field1 = np.nan_to_num(total_field1, nan=0.0, posinf=0.0, neginf=0.0)
        total_field2 = np.nan_to_num(total_field2, nan=0.0, posinf=0.0, neginf=0.0)
        return total_field1, total_field2

    # nominal fields (left/right)
    total_field1_nom, total_field2_nom = compute_total_fields(time_delay)

    # delayed fields (top/bottom)
    if delay_topbottom and delay_amount != 0:
        total_field1_del, total_field2_del = compute_total_fields(time_delay + delay_amount)
    else:
        total_field1_del, total_field2_del = total_field1_nom, total_field2_nom

    # ----------------------------
    # build 4-rectangle geometry with TWO gaps
    # ----------------------------
    half = crystal_size / 2

    # independent center offsets for vertical vs horizontal pair
    Ri_m_x = half + inner_gap_lr / 2   # LEFT/RIGHT shift in X
    Ri_m_y = half + inner_gap_tb / 2   # TOP/BOTTOM shift in Y

    m_top   = (np.abs(X) <= half) & (np.abs(Y - Ri_m_y) <= half)
    m_bot   = (np.abs(X) <= half) & (np.abs(Y + Ri_m_y) <= half)
    m_left  = (np.abs(X + Ri_m_x) <= half) & (np.abs(Y) <= half)
    m_right = (np.abs(X - Ri_m_x) <= half) & (np.abs(Y) <= half)

    m_union = m_top | m_bot | m_left | m_right

    # plot geometry (your helper)
    plot_overlap_geometry(x, y, m_top, m_bot, m_left, m_right,
                          title="Crystal Geometry (TB rotated+delayed; LR normal)")

    # ----------------------------
    # compose Intensity/Gamma EXACTLY like old style,
    # but ADD contributions so overlaps contain both signals
    # ----------------------------
    Gamma_map = np.zeros_like(Rl)
    Intensity = np.zeros_like(Rl)

    # helper to ADD contribution for one region
    def add_region(mask, phi_fn, gamma_fn, a1, a2, f1, f2):
        # bunch contributions (keep separate so we can apply phi weighting like old code)
        g1 = gamma_fn(a1[mask], laser, dcry, f1[mask])
        g2 = gamma_fn(a2[mask], laser, dcry, f2[mask])

        # gamma display map: add (so overlaps show larger total Γ)
        Gamma_map[mask] += (g1 + g2)

        # intensity: old code form, add the two bunch intensities with phi weighting
        Intensity[mask] += (np.sin(phi_fn(a1[mask]))**2) * (np.sin(g1/2)**2)
        Intensity[mask] += (np.sin(phi_fn(a2[mask]))**2) * (np.sin(g2/2)**2)

    # IMPORTANT: order doesn't matter anymore because we ADD.
    # But if you want LR "first", this is LR then TB.
    add_region(m_left,  φ1, Gamma_1, alphab1, alphab2, total_field1_nom, total_field2_nom)
    add_region(m_right, φ1, Gamma_1, alphab1, alphab2, total_field1_nom, total_field2_nom)

    add_region(m_top,   φ2, Gamma_2, alphab1, alphab2, total_field1_del, total_field2_del)
    add_region(m_bot,   φ2, Gamma_2, alphab1, alphab2, total_field1_del, total_field2_del)

    # outside crystals -> 0
    Gamma_map[~m_union] = 0.0
    Intensity[~m_union] = 0.0

    # ----------------------------
    # plots (unchanged)
    # ----------------------------
    fig, axes = plt.subplots(1, 2, figsize=(15, 5), constrained_layout=True)

    im0 = axes[0].imshow(Intensity*100, cmap='plasma', origin='lower',
                         extent=[x.min()*1e3, x.max()*1e3, y.min()*1e3, y.max()*1e3],
                         aspect='equal')
    axes[0].set_title('Intensity')
    axes[0].set_xlabel('x (mm)')
    axes[0].set_ylabel('y (mm)')
    plt.colorbar(im0, ax=axes[0], fraction=0.046, pad=0.04, label='% Transmission at Detector')

    im1 = axes[1].imshow(Gamma_map*1e3, cmap='plasma', origin='lower',
                         extent=[x.min()*1e3, x.max()*1e3, y.min()*1e3, y.max()*1e3],
                         aspect='equal')
    axes[1].set_title(r'Phase Shift (Γ)')
    axes[1].set_xlabel('x (mm)')
    axes[1].set_ylabel('y (mm)')
    plt.colorbar(im1, ax=axes[1], label='mrad', fraction=0.046, pad=0.04)

    plt.show()
    print(f"Maximum Intensity {np.max(Intensity)*100:.3f} %")

    del X, Y, x, y, alphab1, alphab2
    gc.collect()

    return Intensity, Gamma_map, t

In [ ]:
EOS_sim_2b_finite(r01x, r01y, r02x, r02y, time_delay,
                      pix_sizex=3.45e-6, pix_sizey=3.45e-6, magn=2,
                      Q1=800e-12, Q2=800e-12,
                      sig_1=30e-6/3e8, sig_2=30e-6/3e8, delt12=100e-6/c,
                      dcry=100e-6, theta=5,
                      pixelsx=2464, pixelsy=2056,
                      Ri=2.5, Ro=13.2, laser=800e-9, sig_eff_laser=50e-15,
                      cryst_geo=True,
                      # --- geometry params ---
                      crystal_size=10e-3,
                      inner_gap_lr=5e-3,   # gap between LEFT and RIGHT crystals
                      inner_gap_tb=10e-3,   # gap between TOP and BOTTOM crystals
                      # --- delay params ---
                      delay_topbottom=True, delay_amount=100e-6/c,
                      # optional edge masking
                      edges=False)

# EOS_sim_2b_finite(r01x, r01y, r02x, r02y, time_delay,
#                       pix_sizex=3.45e-6, pix_sizey=3.45e-6, magn=2,
#                       Q1=1.6e-9, Q2=0e-12,
#                       sig_1=30e-6/3e8, sig_2=30e-6/3e8, delt12=400e-15,
#                       dcry=100e-6, theta=5,
#                       pixelsx=2464, pixelsy=2056,
#                       Ri=2.5, Ro=13.2, laser=800e-9, sig_eff_laser=50e-15,
#                       cryst_geo=True,
#                       # --- geometry params ---
#                       crystal_size=10e-3,
#                       inner_gap_lr=5e-3,   # gap between LEFT and RIGHT crystals
#                       inner_gap_tb=10e-3,   # gap between TOP and BOTTOM crystals
#                       # --- delay params ---
#                       delay_topbottom=True, delay_amount=100e-6/c,
#                       # optional edge masking
#                       edges=False)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap, BoundaryNorm

def test_crystal_geometry(pixelsx=2856, pixelsy=2856,
                          Ro=13.2,               # mm, only used for plotting extent
                          crystal_size=10e-3,    # m (square side length)
                          inner_gap_lr=2e-3,     # m (gap between LEFT/RIGHT)
                          inner_gap_tb=2e-3,     # m (gap between TOP/BOTTOM)
                          show_outlines=True):
    """
    Build and plot ONLY the 4-rectangle cross geometry with two independent gaps.
    Also plots an overlap-count map to show where 2 crystals overlap.
    """

    # ----------------------------
    # meshgrid on "crystal plane"
    # ----------------------------
    crystx = Ro * 1e-3
    crysty = Ro * 1e-3
    x = np.linspace(-crystx, crystx, pixelsx)
    y = np.linspace(-crysty, crysty, pixelsy)
    X, Y = np.meshgrid(x, y, sparse=False)

    # ----------------------------
    # build 4-rectangle geometry with TWO gaps
    # ----------------------------
    half = crystal_size / 2

    # centers are separated by (crystal_size + gap)
    # so each center is offset by half that: half + gap/2
    Ri_m_x = half + inner_gap_lr / 2  # LEFT/RIGHT shift in X
    Ri_m_y = half + inner_gap_tb / 2  # TOP/BOTTOM shift in Y

    m_top   = (np.abs(X) <= half) & (np.abs(Y - Ri_m_y) <= half)
    m_bot   = (np.abs(X) <= half) & (np.abs(Y + Ri_m_y) <= half)
    m_left  = (np.abs(X + Ri_m_x) <= half) & (np.abs(Y) <= half)
    m_right = (np.abs(X - Ri_m_x) <= half) & (np.abs(Y) <= half)

    # overlap count: 0 (none), 1 (single crystal), 2 (overlap of two)
    overlap_count = (m_top.astype(int) + m_bot.astype(int) +
                     m_left.astype(int) + m_right.astype(int))

    # "label" map: (priority only for visualization)
    # 0 none, 1 top, 2 bottom, 3 left, 4 right
    # NOTE: this is NOT physics ordering—just a display choice.
    label = np.zeros_like(overlap_count, dtype=int)
    label[m_top] = 1
    label[m_bot] = 2
    label[m_left] = 3
    label[m_right] = 4

    # ----------------------------
    # plots
    # ----------------------------
    extent = [x.min()*1e3, x.max()*1e3, y.min()*1e3, y.max()*1e3]  # mm

    fig, axes = plt.subplots(1, 2, figsize=(12, 5), constrained_layout=True)

    # ---- left: label map ----
    cmap = ListedColormap(["black", "tab:blue", "tab:green", "tab:orange", "tab:red"])
    norm = BoundaryNorm([-0.5, 0.5, 1.5, 2.5, 3.5, 4.5], cmap.N)

    im0 = axes[0].imshow(label, origin="lower", extent=extent, cmap=cmap, norm=norm, aspect="equal")
    axes[0].set_title("Crystal labels (display only)")
    axes[0].set_xlabel("x (mm)")
    axes[0].set_ylabel("y (mm)")

    # manual colorbar with names
    cbar0 = plt.colorbar(im0, ax=axes[0], fraction=0.046, pad=0.04, ticks=[0,1,2,3,4])
    cbar0.ax.set_yticklabels(["none", "top", "bottom", "left", "right"])

    # optional outlines to see boundaries precisely
    if show_outlines:
        for m, col in [(m_top, "w"), (m_bot, "w"), (m_left, "w"), (m_right, "w")]:
            axes[0].contour(m.astype(float), levels=[0.5], colors=col, linewidths=0.8, origin="lower", extent=extent)

    # ---- right: overlap count map ----
    im1 = axes[1].imshow(overlap_count, origin="lower", extent=extent, aspect="equal")
    axes[1].set_title("Overlap count (0 none, 1 single, 2 overlap)")
    axes[1].set_xlabel("x (mm)")
    axes[1].set_ylabel("y (mm)")
    plt.colorbar(im1, ax=axes[1], fraction=0.046, pad=0.04, label="count")

    if show_outlines:
        # show where overlap_count>=2
        axes[1].contour((overlap_count >= 2).astype(float), levels=[0.5],
                        colors="w", linewidths=1.0, origin="lower", extent=extent)

    plt.show()

    # quick numeric diagnostics
    pix_area = (x[1]-x[0]) * (y[1]-y[0])  # m^2 per pixel
    area_overlap = np.sum(overlap_count >= 2) * pix_area
    area_union = np.sum(overlap_count >= 1) * pix_area

    print(f"center offsets: Ri_m_x={Ri_m_x*1e3:.3f} mm, Ri_m_y={Ri_m_y*1e3:.3f} mm")
    print(f"union area:     {area_union*1e6:.3f} mm^2")
    print(f"overlap area:   {area_overlap*1e6:.3f} mm^2")
    print(f"overlap fraction of union: {(area_overlap/area_union*100 if area_union>0 else 0):.2f} %")

    return dict(x=x, y=y, X=X, Y=Y,
                m_top=m_top, m_bot=m_bot, m_left=m_left, m_right=m_right,
                overlap_count=overlap_count, label=label)

In [ ]:
test_crystal_geometry(pixelsx=2856, pixelsy=2856,
                          Ro=13.2,               # mm, only used for plotting extent
                          crystal_size=10e-3,    # m (square side length)
                          inner_gap_lr=5e-3,     # m (gap between LEFT/RIGHT)
                          inner_gap_tb=8e-3,     # m (gap between TOP/BOTTOM)
                          show_outlines=True)